## Validation of the filtered Lausanne population against the Microcensus (MZ 2015)
Focus: population structure, activity chains and destinations (filtered data, households living in the study area).

Population definition:
   * Simulation: every agent of the filtered MATSim population, aged `MIN_AGE` or older. Every agent counts equally (weight 1).
   * Microcensus: every target person living in a household inside the study area (`household_ids_in_area.csv`), aged `MIN_AGE` or older, weighted with the MZ person weight (households: `WM`). Population-structure comparisons use all survey days, trip-based comparisons (chains, destinations) weekdays only.

All figures and tables are written to the `results` folder next to this notebook. In all bar charts the simulation is drawn solid and the Microcensus hatched.

### Preparatory Imports

In [ ]:
from pathlib import Path
import csv
import gzip
import re
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, PercentFormatter

import geopandas as gpd
import shapely
import contextily as ctx
from matplotlib.colors import TwoSlopeNorm

In [ ]:
# File Paths
# Relative paths are relative to the folder the notebook runs from (= the folder of this file in Jupyter / VS Code)
NOTEBOOK_DIR = Path.cwd()
OUT_DIR = NOTEBOOK_DIR / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent  # go up one level to ass_2

# MATSIM_POP_FILE = Path('working_rep/filterred_data/lausanne_population_filtered.xml.gz')
# MATSIM_HH_FILE  = Path('working_rep/filterred_data/lausanne_households_filtered.xml.gz')
# HH_IDS_FILE     = Path('working_rep/filterred_data/mz/household_ids_in_area.csv')   # MZ households living in the study area
# MZ_DIR          = Path('Lausanne_10pct/Lausanne_10pct/Mikrozensus/microcensus')
# BOUNDARY_FILE   = Path(r'C:\Users\Sylvain\Documents\GitHub\abmt-2026\ass_2\Lausanne_10pct\Lausanne_10pct\Boundary\LausanneExtent.shp')

# File Paths
MATSIM_POP_FILE = PROJECT_DIR / "Sylvain" / "working_rep" / "filterred_data" / "lausanne_population_filtered.xml.gz"
MATSIM_HH_FILE = PROJECT_DIR / "Sylvain" / "working_rep" / "filterred_data" / "lausanne_households_filtered.xml.gz"
HH_IDS_FILE = PROJECT_DIR / "Sylvain" / "working_rep" / "filterred_data" / "mz" / "household_ids_in_area.csv"
MZ_DIR = PROJECT_DIR / "Lausanne_10pct" / "Lausanne_10pct" / "Mikrozensus" / "microcensus"
BOUNDARY_FILE = ( PROJECT_DIR / "Lausanne_10pct"/ "Lausanne_10pct"/ "Boundary"/ "LausanneExtent.shp")


In [ ]:
# Config
MIN_AGE       = 6                      # persons younger than this are dropped in both sources
MAX_AGE       = 100                    # older persons go into the last, open-ended age group
AGE_BIN_WIDTH = 5
AGE_BINS      = list(range(MIN_AGE, MAX_AGE + 1, AGE_BIN_WIDTH)) + [120]

KEEP_DAYS     = [1, 2, 3, 4, 5]        # MZ survey days for trip-based comparisons (1 = Monday); simulation = average weekday

# household type: MZ W_DEGURBA code / MATSim municipalityType string -> category
HH_CATS            = ['urban', 'suburban', 'rural', 'unknown']
EXCLUDE_UNKNOWN_HH = False
URBAN_MAP_MZ  = {1: 'urban', 2: 'suburban', 3: 'rural', -97: 'unknown'}
URBAN_MAP_SIM = {'urban': 'urban', 'urbancore': 'urban', 'cities': 'urban',
                 'suburban': 'suburban', 'intermediate': 'suburban', 'town': 'suburban', 'towns': 'suburban',
                 'semi-urban': 'suburban', 'semiurban': 'suburban', 'rural': 'rural'}

# activity chains: H home, W work, E education, S shop, L leisure, O other, Ou outside the study area
TOP_N_CHAINS = 12
USE_OUTINGS = True              # MZ: one tour per outing ('ausnr'); False = one tour per person and day
COLLAPSE_RETURN_PAIRS = True    # MZ, only if USE_OUTINGS is False: merge consecutive trips with the same purpose
CLOSE_MODE = 'drop'             # simulation plans not starting/ending at home: 'drop' / 'cyclic' / 'insert'
MZ_HOME = 11
MZ_MAP = {1: 'O', 2: 'W', 3: 'E', 4: 'S', 5: 'O', 6: 'W', 7: 'O', 8: 'L', 9: 'O', 10: 'O',
          11: 'H', 12: 'O', 13: 'O'}
MATSIM_MAP = {'home': 'H', 'work': 'W', 'education': 'E', 'shop': 'S', 'leisure': 'L', 'other': 'O',
              'outside': 'Ou', 'work_secondary': 'W', 'education_secondary': 'E',
              'home_secondary': 'O'}      # judgment call: secondary homes count as 'other'

# destination maps
MAP_PURPOSES = {'W': 'Work', 'E': 'Education', 'S': 'Shopping', 'L': 'Leisure'}
MAP_RUNS = [dict(age_range=(None, None), cell_m=500),     # one figure per entry
            dict(age_range=(6, 25),     cell_m=500)]
MAP_ZOOM = 13
SWISSTOPO_TILES = ('https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-grau/'
                   'default/current/3857/{z}/{x}/{y}.jpeg')

In [ ]:
# %% Boundary ------------------------------------------------------------------
area  = gpd.read_file(BOUNDARY_FILE).to_crs(2056)
shape = shapely.union_all(area.geometry.values)

def inside(x, y):
    """For each point (x, y): True if it lies inside the study area, else False."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return shapely.contains_xy(shape, x, y)       # missing coordinates give False

In [ ]:
# %% Helpers -------------------------------------------------------------------
def read_mz(name, usecols=None, optional=()):
    """MZ csv files: detect separator, tolerate latin-1, case-insensitive columns.
    `optional` columns are kept if the file has them and skipped otherwise."""
    path = MZ_DIR / name
    with open(path, encoding='latin-1') as f:
        sep = csv.Sniffer().sniff(f.readline(), delimiters=',;\t').delimiter
    df = pd.read_csv(path, sep=sep, encoding='latin-1', low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    if usecols:
        wanted = list(usecols) + [c for c in optional if c.lower() in lower]
        df = df[[lower[c.lower()] for c in wanted]]
        df.columns = wanted
    return df


def age_group(age):
    return pd.cut(age.clip(upper=MAX_AGE), AGE_BINS, right=False)


def shares(df, cat, cats):
    """Weighted share of each category of column `cat` (relative to all rows), in the order of `cats`."""
    s = df.groupby(cat, observed=True)['weight'].sum()
    return (s / s.sum()).reindex(cats).fillna(0)


def compare(sim, mz, cat, cats):
    s, m = shares(sim, cat, cats), shares(mz, cat, cats)
    out = pd.DataFrame({'sim': s, 'mz': m, 'diff_pp': (s - m) * 100})
    out['n_sim'] = sim.groupby(cat, observed=True).size().reindex(cats).fillna(0).astype(int)
    out['n_mz']  = mz.groupby(cat, observed=True).size().reindex(cats).fillna(0).astype(int)   # unweighted
    return out.round(4)


# Colours: same palette as the mode-share validation
COLORS = dict(walk='#6cc3a0',           # mint
              bike='#f3c15f',           # butter yellow
              pt='#6f9fe0',             # soft blue
              car='#ee7f62',            # coral
              car_passenger='#9d8ee0',  # lavender
              outside='#c9c8c3')        # light grey
SRC_GREY, GRID_COLOR, TEXT_COLOR = '#bdbcb7', '#e6e5e1', '#52514e'
SEX_COLORS     = dict(male=COLORS['pt'], female=COLORS['car'])
LICENCE_COLORS = dict(yes=COLORS['walk'], no=COLORS['outside'])
URBAN_COLORS   = dict(urban=COLORS['pt'], suburban=COLORS['walk'], rural=COLORS['bike'], unknown=COLORS['outside'])
SEQ_CMAP = LinearSegmentedColormap.from_list('seq', [COLORS['bike'], COLORS['car']])
DIV_CMAP = LinearSegmentedColormap.from_list('div', [COLORS['pt'], '#f7f6f2', COLORS['car']])


def nice_label(v):
    """Readable group label: 'inside' -> 'Inside', interval [6, 11) -> '6–10'."""
    if isinstance(v, pd.Interval):
        lo, hi = int(v.left), int(v.right)
        return f'{lo}+' if hi >= 100 else f'{lo}–{hi - 1}'
    return str(v).replace('_', ' ').capitalize()


def style_axes(ax, grid='y'):
    ax.grid(axis=grid, color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0)


def src_legend():
    """Legend patches: solid = simulation, hatched = Microcensus."""
    return [Patch(facecolor=SRC_GREY, edgecolor='white', label='Simulation'),
            Patch(facecolor=SRC_GREY, edgecolor='white', hatch='///', label='Microcensus')]


def draw_total(ax, s, m, cats, labels=None, colors=None):
    """Horizontal bars per category: simulation on top (solid), Microcensus below (hatched)."""
    labels = labels or {}
    y = np.arange(len(cats))[::-1]               # first category on top
    h, gap = 0.36, 0.04
    for offset, data, hatch in [(+(h + gap) / 2, s, None), (-(h + gap) / 2, m, '///')]:
        ax.barh(y + offset, data[cats], h, color=[colors[c] for c in cats],
                hatch=hatch, edgecolor='white', linewidth=1.2)
        for yi, v in zip(y + offset, data[cats]):
            ax.text(v + 0.006, yi, f'{v:.1%}', va='center', fontsize=12, color=TEXT_COLOR)
    ax.set_yticks(y, [labels.get(c, nice_label(c)) for c in cats])
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_xlim(0, max(s[cats].max(), m[cats].max(), 0.01) * 1.15)
    style_axes(ax, 'x')


def plot_total(sim, mz, cat, cats, fname, labels=None, colors=None, xlabel='Share', footnote=None):
    s, m = shares(sim, cat, cats), shares(mz, cat, cats)
    per = 0.75 if len(cats) <= 6 else 0.55
    fig, ax = plt.subplots(figsize=(7.5, per * len(cats) + 1.2))
    draw_total(ax, s, m, cats, labels, colors)
    ax.set_xlabel(xlabel)
    ax.legend(handles=src_legend(), frameon=False, loc='lower right', handlelength=1.4, handleheight=1.4)
    if footnote:
        fig.text(0.0, -0.02, footnote, fontsize=11, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()


def pyramid_shares(df):
    """Share of all persons per age group and sex (sums to 1 over both sexes)."""
    t = df.pivot_table(index='age_group', columns='sex_', values='weight', aggfunc='sum',
                       fill_value=0, observed=False)
    t = t.reindex(columns=['male', 'female'], fill_value=0)
    return t / t.to_numpy().sum()


def plot_pyramid(sim, mz, fname):
    s, m = pyramid_shares(sim) * 100, pyramid_shares(mz) * 100
    labels = [nice_label(i) for i in s.index]
    y = np.arange(len(labels))                   # youngest group at the bottom
    h, gap = 0.4, 0.03
    fig, ax = plt.subplots(figsize=(8, 0.55 * len(labels) + 1.5))
    for offset, data, hatch in [(+(h + gap) / 2, s, None), (-(h + gap) / 2, m, '///')]:
        for sex, sign in [('male', -1), ('female', 1)]:      # men to the left, women to the right
            ax.barh(y + offset, sign * data[sex], h, color=SEX_COLORS[sex], hatch=hatch,
                    edgecolor='white', linewidth=1.0)
            pad = max(s.to_numpy().max(), m.to_numpy().max()) * 0.015
            for yi, v in zip(y + offset, data[sex]):          # percentage label at the end of each bar
                ax.text(sign * (v + pad), yi, f'{v:.1f}%', va='center',
                        ha='left' if sign > 0 else 'right', fontsize=10, color=TEXT_COLOR)
    ax.axvline(0, color=TEXT_COLOR, linewidth=0.8)
    ax.set_yticks(y, labels)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{abs(v):.0f}%'))
    xmax = max(s.to_numpy().max(), m.to_numpy().max()) * 1.2      # room for the labels
    ax.set_xlim(-xmax, xmax)
    ax.set_xlabel('Share of persons')
    ax.set_ylabel('Age')
    style_axes(ax, 'x')
    handles = [Patch(color=SEX_COLORS['male'], label='Men'), Patch(color=SEX_COLORS['female'], label='Women'),
               Patch(color='none', label='')] + src_legend()
    ax.legend(handles=handles, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left',
              handlelength=1.4, handleheight=1.4)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()


def plot_age_profile(s, m, fname):
    """Single years of age: Microcensus as bars, simulation as a line."""
    ages = np.asarray(s.index)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.bar(ages, m.to_numpy(), width=1.0, color=COLORS['outside'], edgecolor='white', linewidth=0.4,
           label='Microcensus')
    ax.plot(ages, s.to_numpy(), color=COLORS['car'], linewidth=2, label='Simulation')
    ax.set_xlim(ages[0] - 0.5, ages[-1] + 0.5)
    ax.set_xticks(range(int(ages[0]), int(ages[-1]) + 1, 5))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
    ax.set_xlabel('Age')
    ax.set_ylabel('Share of persons')
    style_axes(ax, 'y')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()

### Read and Filter Data

Filter Simulation Output

In [ ]:
# %% 1. Simulation: persons and activities ---------------------------------------
def read_matsim_population(path):
    """Stream the plans file once. Returns
    persons: one row per agent (person, age, sex, hasLicense, has_leg)
    acts:    one row per activity of the selected plan, in plan order (person, type, x, y)"""
    persons, acts = [], []
    with gzip.open(path, 'rb') as f:
        for _, el in ET.iterparse(f, events=('end',)):
            if el.tag != 'person':
                continue
            pid = el.get('id')
            attrs = {a.get('name'): (a.text or '').strip() for a in el.findall('attributes/attribute')}
            plans = el.findall('plan')
            plan = next((p for p in plans if p.get('selected') == 'yes'), plans[0] if plans else None)
            has_leg = plan is not None and plan.find('leg') is not None
            if plan is not None:
                for act in plan.findall('activity'):
                    acts.append((pid, act.get('type', ''), act.get('x'), act.get('y')))
            persons.append((pid, attrs.get('age', el.get('age')),
                            attrs.get('sex', attrs.get('gender', el.get('sex'))),
                            attrs.get('hasLicense'), has_leg))
            el.clear()                           # free memory
    persons = pd.DataFrame(persons, columns=['person', 'age', 'sex', 'hasLicense', 'has_leg'])
    acts = pd.DataFrame(acts, columns=['person', 'type', 'x', 'y'])
    acts[['x', 'y']] = acts[['x', 'y']].apply(pd.to_numeric, errors='coerce')
    return persons, acts


sim_p, sim_acts = read_matsim_population(MATSIM_POP_FILE)

# keep only persons aged MIN_AGE+
sim_p['age'] = pd.to_numeric(sim_p['age'], errors='coerce')
n0 = len(sim_p)
sim_p = sim_p[sim_p['age'] >= MIN_AGE].copy()       # also drops missing ages
print(f'Simulation: {n0:,} persons, {len(sim_p):,} aged >= {MIN_AGE}')

# socio-demographics in the same categories as the MZ
sim_p['age_group'] = age_group(sim_p['age'])
sim_p['age_1y']    = np.floor(sim_p['age']).astype(int)
sim_p['sex_']      = sim_p['sex'].astype(str).str.lower().str[0].map({'m': 'male', 'f': 'female'})
sim_p['licence']   = sim_p['hasLicense'].astype(str).str.lower().map(
    {'yes': 'yes', 'true': 'yes', '1': 'yes', 'no': 'no', 'false': 'no', '0': 'no'})

# Every agent counts equally
sim_p['weight'] = 1.0

In [ ]:
# %% 2. Simulation: households ----------------------------------------------------
def read_matsim_households(path):
    """One row per household with its municipalityType (raw string)."""
    rows, hh = [], None
    with gzip.open(path, 'rb') as f:
        for event, elem in ET.iterparse(f, events=('start', 'end')):
            tag = elem.tag.rsplit('}', 1)[-1]        # strip a possible XML namespace
            if event == 'start':
                if tag == 'household':
                    hh = {'id': elem.get('id'), 'municipalityType': None}
            else:
                if tag == 'attribute' and hh is not None and elem.get('name') == 'municipalityType':
                    hh['municipalityType'] = (elem.text or '').strip()
                elif tag == 'household':
                    rows.append(hh)
                    hh = None
                    elem.clear()
    if not rows:
        raise ValueError('No <household> elements found - check MATSIM_HH_FILE.')
    return pd.DataFrame(rows, columns=['id', 'municipalityType'])


sim_hh = read_matsim_households(MATSIM_HH_FILE)
print(f'Simulation: {len(sim_hh):,} households; municipalityType values:')
print(sim_hh['municipalityType'].fillna('<missing>').value_counts().to_string())

raw = sim_hh['municipalityType'].fillna('').str.lower()
unmapped = sorted(set(raw) - set(URBAN_MAP_SIM) - {''})
if unmapped:
    print(f"WARNING: unmapped municipalityType values counted as 'unknown': {unmapped}")
sim_hh['cat']    = raw.map(URBAN_MAP_SIM).fillna('unknown')
sim_hh['weight'] = 1.0

Filter Microcensus

In [ ]:
# %% 3. Microcensus: persons ------------------------------------------------------
# households living in the study area (list produced in an earlier filtering step)
hh_ids = pd.read_csv(HH_IDS_FILE, encoding='utf-8-sig')
cols = {c.lower(): c for c in hh_ids.columns}
area_hh = pd.to_numeric(hh_ids[cols['hhnr']] if 'hhnr' in cols else hh_ids.iloc[:, 0],
                        errors='coerce').dropna().unique()

# one row per target person (HHNR identifies the person: one per household)
zp = read_mz('zielpersonen.csv', ['HHNR', 'WP', 'alter', 'gesl', 'tag', 'f20400a'])
zp = zp[zp['HHNR'].isin(area_hh)].copy()
print(f'MZ: {len(zp):,} target persons in study-area households')

zp['weight']  = pd.to_numeric(zp['WP'], errors='coerce').fillna(0)     # missing weight -> no contribution
zp['age']     = zp['alter'].where(zp['alter'] >= 0)                    # negative = missing
zp['sex_']    = zp['gesl'].map({1: 'male', 2: 'female'})
zp['licence'] = zp['f20400a'].map({1: 'yes', 2: 'no', -99: 'no'})      # -99 = under 18

zp = zp[zp['age'] >= MIN_AGE].copy()
zp['age_group'] = age_group(zp['age'])
zp['age_1y']    = np.floor(zp['age']).astype(int)

mz_p  = zp                                        # all survey days: structure of the population
mz_pw = zp[zp['tag'].isin(KEEP_DAYS)].copy()      # weekdays only: used together with the trips
print(f'MZ: {len(mz_p):,} persons aged >= {MIN_AGE}, {len(mz_pw):,} of them surveyed on a weekday')

In [ ]:
# %% 4. Microcensus: households ---------------------------------------------------
mz_hh = read_mz('Haushalte.csv', ['HHNR', 'W_DEGURBA'], optional=['WM'])
mz_hh = mz_hh[mz_hh['HHNR'].isin(area_hh)].copy()
print(f'MZ: {len(mz_hh):,} households in the study area')

mz_hh['cat'] = pd.to_numeric(mz_hh['W_DEGURBA'], errors='coerce').map(URBAN_MAP_MZ).fillna('unknown')
if 'WM' in mz_hh.columns:
    mz_hh['weight'] = pd.to_numeric(mz_hh['WM'], errors='coerce').fillna(0)
else:
    print("WARNING: household weight column 'WM' not found -> unweighted")
    mz_hh['weight'] = 1.0

if EXCLUDE_UNKNOWN_HH:
    sim_hh, mz_hh = sim_hh[sim_hh['cat'] != 'unknown'], mz_hh[mz_hh['cat'] != 'unknown']

In [ ]:
# %% 5. Microcensus: trips of the weekday persons ---------------------------------
wege = read_mz('wege.csv', ['HHNR', 'WEGNR', 'wzweck1', 'f51100', 'Z_X_CH1903', 'Z_Y_CH1903'],
               optional=['ausnr'])
wege = wege[wege['f51100'] >= 0]                  # negative departure time = missing / invalid trip

# add person attributes (inner join: only weekday persons of area households)
mz_t = wege.merge(mz_pw[['HHNR', 'weight', 'age', 'sex_']], on='HHNR')
mz_t = mz_t.sort_values(['HHNR', 'WEGNR'])        # trips in order of the day

# destination: CH1903 (LV03) -> LV95 by adding 2,000,000 / 1,000,000; <= 0 = missing
mz_t['dx'] = mz_t['Z_X_CH1903'].where(mz_t['Z_X_CH1903'] > 0) + 2_000_000
mz_t['dy'] = mz_t['Z_Y_CH1903'].where(mz_t['Z_Y_CH1903'] > 0) + 1_000_000
mz_t['dest_in'] = inside(mz_t['dx'], mz_t['dy'])
# 'outside' = destination known and outside the area (trips without coordinates are NOT outside)
mz_t['outside'] = mz_t[['dx', 'dy']].notna().all(axis=1) & ~mz_t['dest_in']
mz_t['letter']  = mz_t['wzweck1'].map(MZ_MAP)     # purpose of the destination, as chain letter
print(f'MZ: {len(mz_t):,} weekday trips, {mz_t["outside"].sum():,} ending outside the study area')

### Comparisons

#### Age and sex (population pyramid)

In [ ]:
# %% 6. Population pyramid, share of all persons per age group and sex ---------------
plot_pyramid(sim_p, mz_p, 'age_pyramid.png')

ps, pm = pyramid_shares(sim_p) * 100, pyramid_shares(mz_p) * 100
pyramid = pd.concat({'sim': ps, 'mz': pm, 'diff_pp': ps - pm}, axis=1).round(3)
pyramid.index = [nice_label(i) for i in pyramid.index]
print(pyramid); pyramid.to_csv(OUT_DIR / 'age_pyramid_table.csv')

In [ ]:
# %% 6b. Share of women and men (weighted) ----------------------------------------
# simulation: every agent weight 1; Microcensus: MZ person weight. Persons with unknown sex are left out.
SEX_CATS = ['male', 'female']
sex_cmp = compare(sim_p, mz_p, 'sex_', SEX_CATS)
sex_cmp['sim_weighted_total'] = sim_p.groupby('sex_')['weight'].sum().reindex(SEX_CATS)
sex_cmp['mz_weighted_total']  = mz_p.groupby('sex_')['weight'].sum().reindex(SEX_CATS).round(1)
sex_cmp.index.name = 'sex'
print(sex_cmp); sex_cmp.to_csv(OUT_DIR / 'sex_share_table.csv')
plot_total(sim_p, mz_p, 'sex_', SEX_CATS, 'sex_share.png',
           labels=dict(male='Men', female='Women'), colors=SEX_COLORS,
           xlabel=f'Share of persons aged {MIN_AGE}+ (weighted)')


#### Age distribution in single years

In [ ]:
# %% 7. Age distribution, 1-year bins ---------------------------------------------
ages = list(range(MIN_AGE, int(max(sim_p['age_1y'].max(), mz_p['age_1y'].max())) + 1))
age_cmp = compare(sim_p, mz_p, 'age_1y', ages)
age_cmp.index.name = 'age'
print(age_cmp.head(10)); age_cmp.to_csv(OUT_DIR / 'age_distribution_1y_table.csv')
plot_age_profile(age_cmp['sim'], age_cmp['mz'], 'age_distribution_1y.png')

#### Driving licence

In [ ]:
# %% 8. Driving licence -----------------------------------------------------------
LICENCE = ['yes', 'no']
licence_cmp = compare(sim_p, mz_p, 'licence', LICENCE)
print(licence_cmp); licence_cmp.to_csv(OUT_DIR / 'driving_licence_table.csv')
plot_total(sim_p, mz_p, 'licence', LICENCE, 'driving_licence.png',
           labels=dict(yes='Has licence', no='No licence'), colors=LICENCE_COLORS,
           xlabel=f'Share of persons aged {MIN_AGE}+')

In [ ]:
# %% 8b. Driving licence by age group ---------------------------------------------
LIC_BINS   = [MIN_AGE, 18, 25, 65, 80, 120]
LIC_LABELS = [f'{MIN_AGE}–17', '18–24', '25–64', '65–79', '80+']
LIC_COLOR  = LICENCE_COLORS['yes']


def licence_by_age(df):
    """Per licence age group: persons with a licence (weighted + raw count), share of the group holding
    a licence, and share of all licence holders falling in the group."""
    d = df[df['licence'].notna()].copy()
    d['lic_age'] = pd.cut(d['age'], LIC_BINS, right=False, labels=LIC_LABELS)
    tot = d.groupby('lic_age', observed=False)['weight'].sum()
    yes = d[d['licence'] == 'yes']
    holders = yes.groupby('lic_age', observed=False)['weight'].sum()
    out = pd.DataFrame({'holders_weighted': holders,
                        'holders_n': yes.groupby('lic_age', observed=False).size(),
                        'group_total_weighted': tot})
    out['rate_in_group'] = out['holders_weighted'] / out['group_total_weighted']
    out['share_of_holders'] = out['holders_weighted'] / out['holders_weighted'].sum()
    return out.fillna(0)


lic_sim, lic_mz = licence_by_age(sim_p), licence_by_age(mz_p)
lic_cmp = lic_sim.add_prefix('sim_').join(lic_mz.add_prefix('mz_'))
lic_cmp['diff_rate_pp'] = (lic_cmp['sim_rate_in_group'] - lic_cmp['mz_rate_in_group']) * 100
lic_cmp['diff_share_pp'] = (lic_cmp['sim_share_of_holders'] - lic_cmp['mz_share_of_holders']) * 100
lic_cmp.index.name = 'age_group'
print(lic_cmp.round(3)); lic_cmp.round(4).to_csv(OUT_DIR / 'driving_licence_by_age_table.csv')

colors = {c: LIC_COLOR for c in LIC_LABELS}
fig, axes = plt.subplots(1, 2, figsize=(14, 0.8 * len(LIC_LABELS) + 1.6))
panels = [('rate_in_group', 'Licence holders as share of their age group'),
          ('share_of_holders', 'Age distribution of all licence holders')]
for ax, (col, title) in zip(axes, panels):
    draw_total(ax, lic_sim[col], lic_mz[col], LIC_LABELS, labels={c: c for c in LIC_LABELS}, colors=colors)
    ax.set_title(title, loc='left', fontsize=15, fontweight='bold')
    ax.set_ylabel('Age group')
    if col == 'rate_in_group':
        ax.set_xlim(0, 1.15)
        ax.set_xlabel('Share of persons with a driving licence')
    else:
        ax.set_xlabel('Share of all persons with a driving licence')
axes[1].legend(handles=src_legend(), frameon=False, loc='lower right', handlelength=1.4, handleheight=1.4)
fig.tight_layout()
fig.savefig(OUT_DIR / 'driving_licence_by_age.png', dpi=200, bbox_inches='tight')
plt.show()


#### Household location type (urban / suburban / rural)

In [ ]:
# %% 9. Household location type ---------------------------------------------------
present = set(sim_hh['cat']) | set(mz_hh['cat'])
hh_cats = [c for c in HH_CATS if c in present]
hh_cmp = compare(sim_hh, mz_hh, 'cat', hh_cats)
print(hh_cmp); hh_cmp.to_csv(OUT_DIR / 'household_urbanity_table.csv')
plot_total(sim_hh, mz_hh, 'cat', hh_cats, 'household_urbanity.png',
           labels=dict(urban='Urban (cities, dense)', suburban='Suburban (towns / suburbs)',
                       rural='Rural (thinly populated)', unknown='Unknown / missing'),
           colors=URBAN_COLORS, xlabel='Share of households')

#### Home municipality
Distribution of the municipality where persons live, compared across three sources: the simulation (location of the home activity), the Microcensus (home coordinates of the household) and the official population statistics of the canton of Vaud (Excel, permanent resident population, 31.12.2025). Only the municipalities that touch the study area are considered, and the three shares are computed relative to the sum over these municipalities. Simulation solid, Microcensus hatched (///), canton data dotted.

In [ ]:
# %% Vertical-bar helper (categories along the x axis, simulation solid, Microcensus hatched) ---------
def plot_vbars(s, m, cats, fname, labels=None, colors=None, ylabel='Share', title=None,
               rotate=0, footnote=None, show_n=None):
    """Grouped vertical bars with percentage labels. s, m = Series of shares (0-1) indexed by category."""
    labels, colors = labels or {}, colors or {}
    x = np.arange(len(cats))
    w = 0.38
    ymax = max(s.reindex(cats).max(), m.reindex(cats).max(), 0.01)
    fig, ax = plt.subplots(figsize=(max(7.5, 0.95 * len(cats) + 2.5), 5.2))
    for offset, data, hatch in [(-w / 2 - 0.01, s, None), (+w / 2 + 0.01, m, '///')]:
        vals = data.reindex(cats).fillna(0).to_numpy()
        ax.bar(x + offset, vals, w, color=[colors.get(c, COLORS['pt']) for c in cats],
               hatch=hatch, edgecolor='white', linewidth=1.2)
        for xi, v in zip(x + offset, vals):
            ax.text(xi, v + ymax * 0.012, f'{v:.1%}', ha='center', va='bottom', fontsize=11, color=TEXT_COLOR)
    ax.set_xticks(x, [labels.get(c, nice_label(c)) for c in cats],
                  rotation=rotate, ha='right' if rotate else 'center')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_ylim(0, ymax * 1.15)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc='left', fontsize=15, fontweight='bold')
    style_axes(ax, 'y')
    ax.legend(handles=src_legend(), frameon=False, loc='upper right', handlelength=1.4, handleheight=1.4)
    if footnote:
        fig.text(0.0, -0.02, footnote, fontsize=11, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()


def plot_vbars3(series, cats, fname, labels=None, colors=None, ylabel='Share', title=None, rotate=0, footnote=None):
    """Grouped vertical bars for three sources. `series` = list of (name, Series of shares, hatch)."""
    labels, colors = labels or {}, colors or {}
    x = np.arange(len(cats))
    n = len(series)
    w = 0.8 / n
    ymax = max(max(s.reindex(cats).max() for _, s, _ in series), 0.01)
    fig, ax = plt.subplots(figsize=(max(8.5, 1.15 * len(cats) + 2.5), 5.4))
    for k, (name, data, hatch) in enumerate(series):
        offset = (k - (n - 1) / 2) * w
        vals = data.reindex(cats).fillna(0).to_numpy()
        ax.bar(x + offset, vals, w * 0.95, color=[colors.get(c, COLORS['pt']) for c in cats],
               hatch=hatch, edgecolor='white', linewidth=1.0)
        for xi, v in zip(x + offset, vals):
            ax.text(xi, v + ymax * 0.012, f'{v:.1%}', ha='center', va='bottom', fontsize=10, color=TEXT_COLOR)
    ax.set_xticks(x, [labels.get(c, nice_label(c)) for c in cats],
                  rotation=rotate, ha='right' if rotate else 'center')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_ylim(0, ymax * 1.15)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, loc='left', fontsize=15, fontweight='bold')
    style_axes(ax, 'y')
    handles = [Patch(facecolor=SRC_GREY, edgecolor='white', hatch=h, label=name) for name, _, h in series]
    ax.legend(handles=handles, frameon=False, loc='upper right', handlelength=1.4, handleheight=1.4)
    if footnote:
        fig.text(0.0, -0.02, footnote, fontsize=11, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# %% 9b. Home municipality of every person --------------------------------------------------------
MUNI_FILE = (PROJECT_DIR / "Sylvain" / "working_rep" / "vaud_limites" / "limites_officielles"
             / "LAD_MO_COMMUNE.shp")
TOP_MUNIS = 12            # municipalities shown individually (largest by average share); the rest -> 'Other'

munis_all = gpd.read_file(MUNI_FILE)
munis_all = munis_all.set_crs(2056) if munis_all.crs is None else munis_all.to_crs(2056)
print('shapefile columns:', [c for c in munis_all.columns if c != 'geometry'])

HOME_NAME_COL = next((c for pref in ('nom_min', 'nom_maj', 'nom', 'name', 'commune', 'nom_commune', 'gemeindename')
                      for c in munis_all.columns if c.lower() == pref), None)
if HOME_NAME_COL is None:      # fall back to the first text column with values - set HOME_NAME_COL by hand if wrong
    HOME_NAME_COL = next(c for c in munis_all.columns
                         if c != 'geometry' and munis_all[c].dtype == object and munis_all[c].notna().any())
print('municipality name column:', HOME_NAME_COL)

# --- Canton Vaud population statistics (Excel) -------------------------------------------------
import unicodedata
CANTON_FILE  = PROJECT_DIR / "Sylvain" / "working_rep" / "Population_communes_origine.xlsx"
CANTON_SHEET = '2025'        # sheet = reference year of the statistics
MIN_OVERLAP  = 0.0           # a municipality counts as 'in the area' if at least this share of its surface lies
                             # inside the study area (0.0 = any overlap; e.g. 0.5 = at least half)


def norm_name(s):
    """Comparable municipality name: no accents / case / punctuation / canton suffix like '(VD)'."""
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    s = re.sub(r'\(.*?\)', '', s)
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()


def read_canton_population(path, sheet):
    """One row per municipality: bfs, name, total resident population (rows with an integer BFS number in col B)."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    rows = []
    for _, r in raw.iterrows():
        bfs = pd.to_numeric(r[1], errors='coerce')
        if pd.notna(bfs) and isinstance(r[2], str) and pd.notna(r[5]):
            rows.append((int(bfs), r[2].strip(), float(r[5])))
    return pd.DataFrame(rows, columns=['bfs', 'name', 'pop'])


canton = read_canton_population(CANTON_FILE, CANTON_SHEET)
print(f'Canton data: {len(canton)} municipalities, total {canton["pop"].sum():,.0f} residents')

# municipalities in the study area (share of their surface inside the area >= MIN_OVERLAP)
overlap = munis_all.geometry.intersection(shape).area / munis_all.geometry.area
in_area = munis_all[(overlap > 0) & (overlap >= MIN_OVERLAP)].copy()
in_area['overlap'] = overlap[in_area.index]

# link shapefile -> Excel: by BFS number if a shapefile column holds them, otherwise by normalised name
bfs_col = next((c for c in munis_all.columns if c != 'geometry'
                and pd.to_numeric(munis_all[c], errors='coerce').isin(canton['bfs']).mean() > 0.5), None)
if bfs_col:
    in_area['canton_pop'] = pd.to_numeric(in_area[bfs_col], errors='coerce').map(canton.set_index('bfs')['pop'])
    print(f'Municipalities linked to the Excel by BFS number (column {bfs_col})')
else:
    in_area['canton_pop'] = in_area[HOME_NAME_COL].map(norm_name).map(canton.assign(k=canton['name'].map(norm_name))
                                                                       .set_index('k')['pop'])
    print(f'Municipalities linked to the Excel by name (column {HOME_NAME_COL})')
if in_area['canton_pop'].isna().any():
    print('WARNING: no canton figure for', in_area.loc[in_area['canton_pop'].isna(), HOME_NAME_COL].tolist())
in_area = in_area.dropna(subset=['canton_pop'])
canton_pop = in_area.groupby(HOME_NAME_COL)['canton_pop'].sum()
canton_share = canton_pop / canton_pop.sum()          # % of the population of all municipalities in the area
print(f'{len(in_area)} municipalities in the study area, {canton_pop.sum():,.0f} residents (canton statistics)')


def muni_of(x, y):
    """Name of the municipality containing each point (NaN if none / missing coordinates)."""
    pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(np.asarray(x, float), np.asarray(y, float)), crs=2056)
    j = gpd.sjoin(pts, munis_all[[HOME_NAME_COL, 'geometry']], how='left', predicate='within')
    j = j[~j.index.duplicated()]
    return j[HOME_NAME_COL].reindex(pts.index).to_numpy()


# --- Simulation: coordinates of the (first) 'home' activity of every person ---
home_act = (sim_acts[sim_acts['type'] == 'home'].dropna(subset=['x', 'y'])
            .drop_duplicates('person')[['person', 'x', 'y']])
sim_home = sim_p[['person', 'weight']].merge(home_act, on='person', how='left')
print(f"Simulation: {sim_home['x'].isna().sum():,} persons without a 'home' activity")
sim_home['muni'] = muni_of(sim_home['x'], sim_home['y'])


# --- Microcensus: home coordinates of the household (LV03 -> LV95) ---
def mz_home_coords():
    """HHNR, x, y (LV95). Uses home coordinates in Haushalte.csv if present, otherwise the destination
    of the 'return home' trips (wzweck1 == 11) in wege.csv."""
    hh = read_mz('Haushalte.csv')
    low = {c.lower(): c for c in hh.columns}
    for cx, cy in [('w_x_ch1903', 'w_y_ch1903'), ('wx_ch1903', 'wy_ch1903'), ('hh_x_ch1903', 'hh_y_ch1903'),
                   ('w_x_lv95', 'w_y_lv95')]:
        if cx in low and cy in low:
            out = hh[['HHNR', low[cx], low[cy]]].set_axis(['HHNR', 'x', 'y'], axis=1)
            print(f'MZ home coordinates from Haushalte.csv ({low[cx]}, {low[cy]})')
            break
    else:
        print('No home coordinates in Haushalte.csv -> using the destination of the return-home trips (wege.csv)')
        w = read_mz('wege.csv', ['HHNR', 'wzweck1', 'Z_X_CH1903', 'Z_Y_CH1903'])
        w = w[(w['wzweck1'] == MZ_HOME) & (w['Z_X_CH1903'] > 0) & (w['Z_Y_CH1903'] > 0)]
        out = (w.drop_duplicates('HHNR')[['HHNR', 'Z_X_CH1903', 'Z_Y_CH1903']]
                .set_axis(['HHNR', 'x', 'y'], axis=1))
    out = out.apply(pd.to_numeric, errors='coerce')
    out = out[(out['x'] > 0) & (out['y'] > 0)].copy()
    lv03 = out['x'] < 1_000_000                       # LV03 values -> add the LV95 offset
    out.loc[lv03, 'x'] += 2_000_000
    out.loc[lv03, 'y'] += 1_000_000
    return out.drop_duplicates('HHNR')


mz_home = mz_p[['HHNR', 'weight']].merge(mz_home_coords(), on='HHNR', how='left')
print(f"MZ: {mz_home['x'].isna().sum():,} persons without home coordinates")
mz_home['muni'] = muni_of(mz_home['x'], mz_home['y'])

# --- keep persons with a municipality; shares are relative to those ---
n_s, n_m = sim_home['muni'].isna().sum(), mz_home['muni'].isna().sum()
if n_s or n_m:
    print(f'Dropped (no municipality found): simulation {n_s:,}, MZ {n_m:,}')
sim_home, mz_home = sim_home.dropna(subset=['muni']), mz_home.dropna(subset=['muni'])
# only municipalities that are part of the study area (same set as the canton figures)
sim_home = sim_home[sim_home['muni'].isin(canton_pop.index)]
mz_home  = mz_home[mz_home['muni'].isin(canton_pop.index)]
if sim_home.empty or mz_home.empty:
    raise ValueError(f"No home matched a municipality - check HOME_NAME_COL ({HOME_NAME_COL}) and the shapefile CRS.")

# --- full table, then top N + 'Other' ---
all_munis = sorted(set(sim_home['muni']) | set(mz_home['muni']) | set(canton_share.index))
sim_sh, mz_sh = shares(sim_home, 'muni', all_munis), shares(mz_home, 'muni', all_munis)
can_sh = canton_share.reindex(all_munis).fillna(0)
avg = (sim_sh + mz_sh + can_sh) / 3
all_munis = list(avg.sort_values(ascending=False).index)

home_full = compare(sim_home, mz_home, 'muni', all_munis)
home_full['canton'] = can_sh.reindex(all_munis).round(4)
home_full['diff_sim_canton_pp'] = ((home_full['sim'] - home_full['canton']) * 100).round(4)
home_full['diff_mz_canton_pp']  = ((home_full['mz'] - home_full['canton']) * 100).round(4)
home_full['canton_pop'] = canton_pop.reindex(all_munis)
home_full.index.name = 'municipality'
home_full.to_csv(OUT_DIR / 'home_municipality_table.csv')

top_m = all_munis[:TOP_MUNIS]
muni_cats = top_m + (['Other'] if len(all_munis) > TOP_MUNIS else [])
sim_g = sim_home.assign(muni=sim_home['muni'].where(sim_home['muni'].isin(top_m), 'Other'))
mz_g  = mz_home.assign(muni=mz_home['muni'].where(mz_home['muni'].isin(top_m), 'Other'))
can_g = can_sh.groupby(can_sh.index.where(can_sh.index.isin(top_m), 'Other')).sum()
home_cmp = compare(sim_g, mz_g, 'muni', muni_cats)
home_cmp['canton'] = can_g.reindex(muni_cats).round(4)
home_cmp.index.name = 'municipality'
print(home_cmp)
home_cmp.to_csv(OUT_DIR / 'home_municipality_table_top.csv')

plot_vbars3([('Simulation', shares(sim_g, 'muni', muni_cats), None),
             ('Microcensus', shares(mz_g, 'muni', muni_cats), '///'),
             (f'Canton Vaud ({CANTON_SHEET})', can_g, '...')],
            muni_cats, 'home_municipality.png',
            labels={c: c for c in muni_cats},
            colors={**{c: COLORS['pt'] for c in top_m}, 'Other': COLORS['outside']},
            ylabel='Share of residents', rotate=45,
            title='Municipality of residence: Simulation vs Microcensus vs Canton Vaud',
            footnote=f"Only the {len(canton_pop)} municipalities in the study area; shares relative to their sum. "
                     f"Top {TOP_MUNIS} by average share, 'Other' = remaining {max(len(all_munis) - TOP_MUNIS, 0)}. "
                     f"Simulation / MZ: persons aged {MIN_AGE}+; canton: all ages.")


#### Public-transport quality (ÖV-Güteklasse)
Household attribute `ovgk` (simulation) vs `W_OeV_KLASSE` (Microcensus, households weighted with `WM`). A = very good, B = good, C = mediocre, D = low, none = marginal or no PT service.

A third bar shows the ARE ÖV-Güteklasse looked up (geo.admin.ch API, layer `ch.are.gueteklassen_oev`) at the Microcensus household home coordinates (`W_X_CH1903`, `W_Y_CH1903` in Haushalte.csv), using the same household weights `WM`. Results are cached in `results/are_gueteklasse_lookup.csv`. Requires internet access on the first run.

A fourth bar does the same for the simulation: the ARE class at the household home coordinates (`home_coordinate_x` / `home_coordinate_y` person attributes of the population file, one coordinate per household).


In [ ]:
# %% 9c. Public-transport quality (Gueteklasse) of the household ---------------------------------
def read_matsim_hh_attr(path, attr):
    """One row per household: id and the raw value of household attribute `attr` (None if absent)."""
    rows, hh = [], None
    with gzip.open(path, 'rb') as f:
        for event, elem in ET.iterparse(f, events=('start', 'end')):
            tag = elem.tag.rsplit('}', 1)[-1]
            if event == 'start':
                if tag == 'household':
                    hh = {'id': elem.get('id'), 'value': None}
            else:
                if tag == 'attribute' and hh is not None and elem.get('name') == attr:
                    hh['value'] = (elem.text or '').strip()
                elif tag == 'household':
                    rows.append(hh); hh = None; elem.clear()
    return pd.DataFrame(rows, columns=['id', 'value'])


OV_CATS   = ['A', 'B', 'C', 'D', 'none']
OV_LABELS = {'A': 'Class A\nvery good', 'B': 'Class B\ngood', 'C': 'Class C\nmediocre',
             'D': 'Class D\nlow', 'none': 'No class\nmarginal / none', 'unknown': 'Unknown'}
OV_COLORS = {'A': '#4f7fc9', 'B': COLORS['pt'], 'C': '#9fbfea', 'D': '#c5d8f2',
             'none': COLORS['outside'], 'unknown': GRID_COLOR}
MZ_OV_MAP = {1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'none'}

# --- Simulation: household attribute 'ovgk' ---
ov_raw = read_matsim_hh_attr(MATSIM_HH_FILE, 'ovgk')
print('Simulation ovgk, raw values:'); print(ov_raw['value'].fillna('<missing>').value_counts().to_string())
sim_ov = ov_raw.assign(cat=ov_raw['value'].str.upper().map(lambda v: v if v in ('A', 'B', 'C', 'D') else 'none'),
                       weight=1.0)
sim_ov.loc[ov_raw['value'].isna(), 'cat'] = 'unknown'          # attribute missing on the household

# --- Microcensus: W_OeV_KLASSE in Haushalte.csv, household weight WM ---
mz_ov_raw = read_mz('Haushalte.csv')
low = {c.lower(): c for c in mz_ov_raw.columns}
mz_ov = mz_ov_raw[mz_ov_raw[low['hhnr']].isin(area_hh)].copy()
mz_ov['cat'] = pd.to_numeric(mz_ov[low['w_oev_klasse']], errors='coerce').map(MZ_OV_MAP).fillna('unknown')
mz_ov['weight'] = pd.to_numeric(mz_ov[low['wm']], errors='coerce').fillna(0) if 'wm' in low else 1.0
print(f'\nMZ: {len(mz_ov):,} households in the study area')

# --- Microcensus addresses: ARE Gueteklasse looked up at the household home coordinates ---------------
# Source: https://opendata.swiss/de/dataset/ov-guteklassen-are1 (layer ch.are.gueteklassen_oev, geo.admin.ch REST API).
# Home coordinates = W_X_CH1903 / W_Y_CH1903 of Haushalte.csv (converted to LV95 by mz_home_coords()).
import json, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor

ARE_IDENTIFY_URL = 'https://api3.geo.admin.ch/rest/services/api/MapServer/identify'
ARE_LAYER        = 'ch.are.gueteklassen_oev'
ARE_CACHE        = OUT_DIR / 'are_gueteklasse_lookup.csv'     # one row per looked-up coordinate (re-runs are instant)
ARE_CLASS_FIELD  = None      # attribute holding the class; None = auto-detect (see the printout of the first answer)
_are_printed     = []


def _are_class_from_attrs(attrs):
    """Class letter (A-D) from the attributes of one identified polygon; None if not found."""
    keys = [ARE_CLASS_FIELD] if ARE_CLASS_FIELD else [k for k in attrs if 'klass' in k.lower() or 'class' in k.lower()]
    for k in keys + list(attrs):
        v = str(attrs.get(k, '')).strip().upper()          # e.g. 'D - geringe Erschliessung' -> 'D'
        if v[:1] in ('A', 'B', 'C', 'D', 'E', 'F') and (len(v) == 1 or v[1:2] in (' ', '-')):
            return v[0]
    return None


def are_class_at(x, y, retries=3):
    """ARE Gueteklasse at an LV95 point: 'A'-'D', 'none' (no polygon = marginal / no PT service) or 'error'."""
    params = {'geometryType': 'esriGeometryPoint', 'geometry': f'{x},{y}', 'sr': 2056, 'layers': f'all:{ARE_LAYER}',
              'tolerance': 0, 'mapExtent': f'{x - 50},{y - 50},{x + 50},{y + 50}', 'imageDisplay': '100,100,96',
              'returnGeometry': 'false', 'lang': 'en'}
    url = ARE_IDENTIFY_URL + '?' + urllib.parse.urlencode(params)
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=20) as r:
                results = json.load(r).get('results', [])
            break
        except Exception:
            if attempt == retries - 1:
                return 'error'
    if not results:
        return 'none'
    if not _are_printed:
        _are_printed.append(1); print('First ARE answer, attributes:', results[0].get('attributes'))
    found = [c for c in (_are_class_from_attrs(r.get('attributes', {})) for r in results) if c]
    if not found:
        return 'error'                                   # polygon found but no class field recognised -> set ARE_CLASS_FIELD
    best = min(found)                                    # overlapping polygons: keep the best class
    return best if best in ('A', 'B', 'C', 'D') else 'none'


def lookup_are_classes(xy):
    """DataFrame x, y, cls for all unique (rounded) LV95 coordinates in `xy`; cached on disk."""
    xy = xy.round(0).astype(int).drop_duplicates()
    cache = pd.read_csv(ARE_CACHE) if ARE_CACHE.exists() else pd.DataFrame(columns=['x', 'y', 'cls'])
    todo = xy.merge(cache[cache['cls'] != 'error'][['x', 'y']], on=['x', 'y'], how='left', indicator=True)
    todo = todo[todo['_merge'] == 'left_only'][['x', 'y']]
    if len(todo):
        print(f'Querying the ARE service for {len(todo):,} coordinates ...')
        with ThreadPoolExecutor(max_workers=8) as ex:
            todo['cls'] = list(ex.map(are_class_at, todo['x'], todo['y']))
        cache = pd.concat([cache[cache['cls'] != 'error'], todo]).drop_duplicates(['x', 'y'], keep='last')
        cache.to_csv(ARE_CACHE, index=False)
    return cache


mz_ov_xy = mz_home_coords()
mz_ov_xy = mz_ov_xy[pd.to_numeric(mz_ov_xy['HHNR'], errors='coerce')
                    .isin(pd.to_numeric(mz_ov[low['hhnr']], errors='coerce'))]      # study-area households only
print(f'Looking up {len(mz_ov_xy):,} study-area households (not the whole Microcensus)')
are_cache = lookup_are_classes(mz_ov_xy[['x', 'y']])
mz_addr = (mz_ov[[low['hhnr'], 'weight']].rename(columns={low['hhnr']: 'HHNR'})
           .merge(mz_ov_xy.assign(x=lambda d: d['x'].round(0).astype(int), y=lambda d: d['y'].round(0).astype(int)),
                  on='HHNR', how='left')
           .merge(are_cache, on=['x', 'y'], how='left'))
mz_addr['cat'] = mz_addr['cls'].fillna('unknown').replace('error', 'unknown')    # no coordinates / failed lookup -> unknown
print('\nMZ household addresses, ARE class counts:'); print(mz_addr['cat'].value_counts().to_string())

# --- Simulation addresses: ARE Gueteklasse looked up at the household home coordinates ----------------
# Home coordinates = person attributes home_coordinate_x / home_coordinate_y of the population file (LV95);
# one coordinate per household (first member that has one), households as in the households file.
def read_matsim_home_coords(path):
    """person id -> home_coordinate_x / home_coordinate_y (attributes of the <person>)."""
    rows = []
    with gzip.open(path, 'rb') as f:
        for _, el in ET.iterparse(f, events=('end',)):
            if el.tag != 'person':
                continue
            a = {x.get('name'): (x.text or '').strip() for x in el.findall('attributes/attribute')}
            rows.append((el.get('id'), a.get('home_coordinate_x'), a.get('home_coordinate_y')))
            el.clear()
    out = pd.DataFrame(rows, columns=['person', 'x', 'y'])
    out[['x', 'y']] = out[['x', 'y']].apply(pd.to_numeric, errors='coerce')
    return out


def read_matsim_hh_members(path):
    """household id -> member person ids, from <household><members><personId refId=.../></members>."""
    rows, hh = [], None
    with gzip.open(path, 'rb') as f:
        for event, el in ET.iterparse(f, events=('start', 'end')):
            t = el.tag.rsplit('}', 1)[-1]
            if event == 'start':
                if t == 'household':
                    hh = el.get('id')
            elif t == 'personId' and hh is not None:
                rows.append((hh, el.get('refId')))
            elif t == 'household':
                hh = None; el.clear()
    return pd.DataFrame(rows, columns=['id', 'person'])


sim_home_xy = read_matsim_home_coords(MATSIM_POP_FILE)
sim_hh_xy = (read_matsim_hh_members(MATSIM_HH_FILE).merge(sim_home_xy, on='person', how='left')
             .dropna(subset=['x', 'y']).drop_duplicates('id')[['id', 'x', 'y']])
sim_hh_xy[['x', 'y']] = sim_hh_xy[['x', 'y']].round(0).astype(int)
print(f"\nSimulation: {len(sim_hh_xy):,} of {len(sim_ov):,} households have a home coordinate "
      f"({len(sim_hh_xy[['x', 'y']].drop_duplicates()):,} unique points)")
are_cache = lookup_are_classes(sim_hh_xy[['x', 'y']])
sim_addr = (sim_ov[['id', 'weight']].merge(sim_hh_xy, on='id', how='left').merge(are_cache, on=['x', 'y'], how='left'))
sim_addr['cat'] = sim_addr['cls'].fillna('unknown').replace('error', 'unknown')
print('Simulation household addresses, ARE class counts:'); print(sim_addr['cat'].value_counts().to_string())

ov_cats = OV_CATS + (['unknown'] if 'unknown' in set(sim_ov['cat']) | set(mz_ov['cat']) | set(mz_addr['cat']) | set(sim_addr['cat']) else [])
ov_cmp = compare(sim_ov, mz_ov, 'cat', ov_cats)
ov_cmp['mz_addr'] = shares(mz_addr, 'cat', ov_cats).round(4)                       # MZ addresses -> ARE class
ov_cmp['diff_addr_pp'] = ((ov_cmp['sim'] - ov_cmp['mz_addr']) * 100).round(4)
ov_cmp['n_mz_addr'] = mz_addr.groupby('cat').size().reindex(ov_cats).fillna(0).astype(int)
ov_cmp['sim_addr'] = shares(sim_addr, 'cat', ov_cats).round(4)                    # MATSim addresses -> ARE class
ov_cmp['diff_sim_vs_sim_addr_pp'] = ((ov_cmp['sim'] - ov_cmp['sim_addr']) * 100).round(4)
ov_cmp['n_sim_addr'] = sim_addr.groupby('cat').size().reindex(ov_cats).fillna(0).astype(int)
ov_cmp.index.name = 'oev_class'
print(ov_cmp); ov_cmp.to_csv(OUT_DIR / 'oev_gueteklasse_table.csv')

plot_vbars3([('Simulation', shares(sim_ov, 'cat', ov_cats), None),
             ('Microcensus (W_OeV_KLASSE)', shares(mz_ov, 'cat', ov_cats), '///'),
             ('Simulation addresses (ARE map)', shares(sim_addr, 'cat', ov_cats), '.'),   # sparse dots
             ('Microcensus addresses (ARE map)', shares(mz_addr, 'cat', ov_cats), '++')], # grid
            ov_cats, 'oev_gueteklasse.png', labels=OV_LABELS, colors=OV_COLORS, ylabel='Share of households',
            title='Public-transport quality (Gueteklasse): Simulation vs Microcensus vs ARE at addresses',
            footnote='Dotted / grid bars: ARE Gueteklasse (geo.admin.ch, ch.are.gueteklassen_oev) at the household home coordinates '
                     'of the simulation (home_coordinate_x/y) and of the Microcensus (W_X/Y_CH1903, weighted with WM).')


In [ ]:
#### Income distribution (income class)
# Paste into the notebook after the "Household location type" section.
# Uses helpers already defined above: read_mz, compare, shares, style_axes, src_legend, COLORS, TEXT_COLOR, OUT_DIR.

# %% 10. Income class: simulation vs. Microcensus ---------------------------------
MZ_INCOME_LABELS = {1: '< 2k', 2: '2–4k', 3: '4–6k', 4: '6–8k', 5: '8–10k',
                    6: '10–12k', 7: '12–14k', 8: '14–16k', 9: '> 16k'}   # CHF per month (MZ F20601A)
INCOME_COLOR = COLORS['pt']


def read_matsim_income(path, tag):
    """Stream a MATSim xml(.gz) and return one row per <tag> element (household / person)
    with its integer attribute `incomeClass` (NaN if the element has none)."""
    rows, cur = [], None
    with gzip.open(path, 'rb') as f:
        for event, el in ET.iterparse(f, events=('start', 'end')):
            t = el.tag.rsplit('}', 1)[-1]
            if event == 'start':
                if t == tag:
                    cur = {'id': el.get('id'), 'income': None}
            else:
                if t == 'attribute' and cur is not None and el.get('name') == 'incomeClass':
                    cur['income'] = (el.text or '').strip()
                elif t == tag and cur is not None:
                    rows.append(cur)
                    cur = None
                    el.clear()
    df = pd.DataFrame(rows, columns=['id', 'income'])
    df['income'] = pd.to_numeric(df['income'], errors='coerce')
    return df


# Simulation: income class is stored on the household if present there, otherwise on the person
sim_inc = read_matsim_income(MATSIM_HH_FILE, 'household')
unit = 'households'
if sim_inc['income'].notna().sum() == 0:
    sim_inc = read_matsim_income(MATSIM_POP_FILE, 'person')
    unit = 'persons'
    sim_inc = sim_inc[sim_inc['id'].isin(sim_p['person'])]          # same population as the rest (age >= MIN_AGE)
print(f'Simulation: incomeClass read from {unit}; {sim_inc["income"].notna().sum():,} of {len(sim_inc):,} have a value')
sim_inc['income'] = sim_inc['income'] + 1      # MATSim classes are 0-8, MZ classes 1-9 -> shift by one
sim_inc['weight'] = 1.0

# Microcensus: household income (F20601A), study-area households only, weighted with WM
mz_inc = read_mz('Haushalte.csv', ['HHNR'], optional=['F20601A', 'F20601', 'WM'])
mz_inc = mz_inc[mz_inc['HHNR'].isin(area_hh)].copy()
inc_col = 'F20601A' if 'F20601A' in mz_inc.columns else 'F20601'
mz_inc['income'] = pd.to_numeric(mz_inc[inc_col], errors='coerce')
mz_inc['weight'] = pd.to_numeric(mz_inc['WM'], errors='coerce').fillna(0) if 'WM' in mz_inc.columns else 1.0
print(f'MZ: {len(mz_inc):,} households; answers of {inc_col} (raw codes, negative = missing):')
print(mz_inc['income'].value_counts().sort_index().to_string())

# valid answers only (MZ: -99 / -98 / -97 are dropped; shares are relative to valid answers)
sim_inc = sim_inc[sim_inc['income'].notna()].copy()
mz_inc = mz_inc[mz_inc['income'] >= 1].copy()
sim_inc['income'] = sim_inc['income'].astype(int)
mz_inc['income'] = mz_inc['income'].astype(int)

INCOME_CATS = sorted(set(MZ_INCOME_LABELS) | set(sim_inc['income']) | set(mz_inc['income']))
income_cmp = compare(sim_inc, mz_inc, 'income', INCOME_CATS)
income_cmp.index.name = 'income_class'
print(income_cmp); income_cmp.to_csv(OUT_DIR / 'income_distribution_table.csv')

# Figure: distribution of the income classes (simulation solid, Microcensus hatched)
s, m = income_cmp['sim'], income_cmp['mz']
x = np.arange(len(INCOME_CATS))
w = 0.38
fig, ax = plt.subplots(figsize=(max(8, 0.95 * len(INCOME_CATS) + 2.5), 5.2))
for offset, data, hatch in [(-w / 2 - 0.01, s, None), (+w / 2 + 0.01, m, '///')]:
    ax.bar(x + offset, data.to_numpy(), w, color=INCOME_COLOR, hatch=hatch, edgecolor='white', linewidth=1.2)
    for xi, v in zip(x + offset, data.to_numpy()):
        ax.text(xi, v + 0.004, f'{v:.1%}', ha='center', va='bottom', fontsize=10, color=TEXT_COLOR)
ax.set_xticks(x, [f'{c}\n{MZ_INCOME_LABELS[c]}' if c in MZ_INCOME_LABELS else str(c) for c in INCOME_CATS])
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_ylim(0, max(s.max(), m.max()) * 1.15)
ax.set_xlabel('Income class (MZ: monthly household income in CHF)')
ax.set_ylabel(f'Share of {unit} with a valid answer')
style_axes(ax, 'y')
ax.legend(handles=src_legend(), frameon=False, loc='upper right', handlelength=1.4, handleheight=1.4)
fig.tight_layout()
fig.savefig(OUT_DIR / 'income_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

#### Activity chains
Home-based tours written as letter strings, e.g. `HWH` = home - work - home. Persons without any tour count as `H`. Only basic activities (work, education, shop, leisure, other) and `Ou` = activity outside the study area. Shares are computed within each source (and within each sex). Only the `TOP_N_CHAINS` most frequent chains are shown individually, all others are grouped in `Other`; men and women use the same chains in the same order as the total figure.

MZ purpose codes (`wzweck1`): 1 change of mode / park car, 2 work, 3 education, 4 shopping, 5 errands / services, 6 business, 7 service trip, 8 leisure, 9 accompanying (children), 10 accompanying / service (others), 11 return home, 12 other, 13 border crossing.

In [ ]:
# %% 10. Chain helpers (shared by simulation and MZ) -----------------------------
def merge_outside(chain):
    """Collapse consecutive 'Ou' stops into one: HOuOuOuH -> HOuH."""
    return re.sub(r'(?:Ou)+', 'Ou', chain)


def split_into_tours(codes):
    """Cut a sequence of letters at every 'H' into complete home-to-home tours; anything before
    the first or after the last H is discarded. ['H','W','H','S','L','H'] -> ['HWH','HSLH']"""
    tours, current, started = [], [], False
    for code in codes:
        if code == 'H':
            if started and current:
                tours.append('H' + ''.join(current) + 'H')
            current, started = [], True
        elif started:
            current.append(code)
    return tours


def plan_tours(seq, mode):
    """One daily letter sequence -> (tours, status); status 'home' (never leaves home),
    'excluded' (no usable tour), 'ok' or 'closed' (plan repaired according to `mode`)."""
    if all(c == 'H' for c in seq):
        return [], 'home'
    if 'H' not in seq:
        return [], 'excluded'
    if seq[0] == 'H' and seq[-1] == 'H':
        return split_into_tours(seq), 'ok'
    if mode == 'drop':
        tours = split_into_tours(seq)
        return tours, ('ok' if tours else 'excluded')
    if mode == 'cyclic':
        first_home = seq.index('H')
        seq = seq[first_home:] + seq[:first_home] + ['H']
    else:                                          # 'insert'
        if seq[0] != 'H':
            seq = ['H'] + seq
        if seq[-1] != 'H':
            seq = seq + ['H']
    return split_into_tours(seq), 'closed'


def to_letters(purposes, outside_flags):
    """MZ purpose codes -> chain letters. Home trips and unknown codes are dropped,
    trips ending outside the study area become 'Ou'."""
    letters = []
    for code, outside in zip(purposes, outside_flags):
        if code == MZ_HOME:
            continue
        letter = 'Ou' if outside else MZ_MAP.get(code)
        if letter:
            letters.append(letter)
    return letters


def person_tours(trips):
    """Tour strings of one MZ person from its ordered trips."""
    if USE_OUTINGS and 'ausnr' in trips.columns:
        tours = []
        for _, outing in trips.groupby('ausnr', sort=False):
            acts = to_letters(outing['wzweck1'].iloc[:-1], outing['outside'].iloc[:-1])   # last trip = back home
            if acts:
                tours.append(merge_outside('H' + ''.join(acts) + 'H'))
        return tours
    pairs = list(zip(trips['wzweck1'], trips['outside']))
    if COLLAPSE_RETURN_PAIRS:                      # one tour per day, consecutive duplicates merged
        kept = [p for i, p in enumerate(pairs) if i == 0 or p[0] != pairs[i - 1][0]]
        acts = to_letters(*zip(*kept)) if kept else []
        return [merge_outside('H' + ''.join(acts) + 'H')] if acts else []
    seq = ['H']                                    # split the day at every explicit home trip
    for code, out in pairs:
        if code == MZ_HOME:
            seq.append('H')
        else:
            seq.extend(to_letters([code], [out]))
    return [merge_outside(t) for t in split_into_tours(seq)]


def chain_order(sim, mz):
    """All chains, most frequent in the MZ first."""
    rank = mz.groupby('chain')['weight'].sum().sort_values(ascending=False)
    return list(rank.index) + [c for c in sim['chain'].unique() if c not in rank.index]

In [ ]:
# %% 11. Chains: one row per tour -----------------------------------------------------
# simulation
seqs = (sim_acts.assign(letter=sim_acts['type'].map(MATSIM_MAP)).dropna(subset=['letter'])
        .groupby('person', sort=False)['letter'].agg(list))
rows, status = [], Counter()
for pid, sex, has_leg in zip(sim_p['person'], sim_p['sex_'], sim_p['has_leg']):
    tours, st = plan_tours(seqs.get(pid, []), CLOSE_MODE) if has_leg else ([], 'home')
    status[st] += 1
    if tours:
        rows += [(pid, sex, merge_outside(t)) for t in tours]
    elif st == 'home':
        rows.append((pid, sex, 'H'))               # 'excluded' persons are not counted
sim_chains = pd.DataFrame(rows, columns=['person', 'sex_', 'chain'])
sim_chains['weight'] = 1.0
print('Simulation plan status:', dict(status))

# Microcensus (weighted with the person weight)
rows = [(hh, t) for hh, g in mz_t.groupby('HHNR') for t in person_tours(g)]
mz_chains = pd.DataFrame(rows, columns=['HHNR', 'chain']).merge(mz_pw[['HHNR', 'sex_', 'weight']], on='HHNR')
stay = mz_pw[~mz_pw['HHNR'].isin(mz_chains['HHNR'])][['HHNR', 'sex_', 'weight']].assign(chain='H')
mz_chains = pd.concat([mz_chains, stay], ignore_index=True)
print(f'MZ: {len(stay):,} weekday persons without any tour (counted as H)')
print(f'tours: simulation {len(sim_chains):,}, MZ {len(mz_chains):,} (unweighted)')

In [ ]:
# %% 12. Chain shares, all persons ------------------------------------------------------
CHAIN_NOTE = 'H home, W work, E education, S shop, L leisure, O other, Ou outside the study area'
OTHER = 'Other'                                   # label for every chain outside the top N
OTHER_COLOR = COLORS['outside']


def group_other(df, top):
    """Copy of df where every chain not in `top` is renamed to 'Other'."""
    out = df.copy()
    out['chain'] = out['chain'].where(out['chain'].isin(top), OTHER)
    return out


def unique_chain_summary(sim, mz):
    """Number of distinct chains (unweighted) in each source, shared and exclusive ones."""
    s, m = set(sim['chain']), set(mz['chain'])
    return {'simulation': len(s), 'microcensus': len(m), 'in both': len(s & m),
            'only simulation': len(s - m), 'only microcensus': len(m - s), 'union': len(s | m)}


# --- unique chains (all persons and by sex) ---
uniq = {'all': unique_chain_summary(sim_chains, mz_chains)}
for sex in ['male', 'female']:
    uniq[sex] = unique_chain_summary(sim_chains[sim_chains['sex_'] == sex],
                                     mz_chains[mz_chains['sex_'] == sex])
uniq_df = pd.DataFrame(uniq).T
print('Number of unique activity chains:'); print(uniq_df)
uniq_df.to_csv(OUT_DIR / 'chains_unique_counts.csv')

# --- full comparison table (every chain) ---
chains_all = chain_order(sim_chains, mz_chains)
chain_cmp = compare(sim_chains, mz_chains, 'chain', chains_all)
chain_cmp.index.name = 'chain'
chain_cmp.to_csv(OUT_DIR / 'chains_total.csv')

# --- top N + 'Other' (this order is reused for the men / women figure) ---
top = chains_all[:TOP_N_CHAINS]
chain_cats = top + [OTHER]
sim_top, mz_top = group_other(sim_chains, top), group_other(mz_chains, top)
chain_cmp_top = compare(sim_top, mz_top, 'chain', chain_cats)
chain_cmp_top.index.name = 'chain'
print(chain_cmp_top); chain_cmp_top.to_csv(OUT_DIR / 'chains_top_other.csv')

n_other_sim = uniq['all']['simulation'] - len(set(sim_chains['chain']) & set(top))
n_other_mz  = uniq['all']['microcensus'] - len(set(mz_chains['chain']) & set(top))
note = (f"{CHAIN_NOTE}\n'Other' = all remaining chains ({n_other_sim:,} unique in simulation, {n_other_mz:,} in MZ). "
        f"Unique chains in total: simulation {uniq['all']['simulation']:,}, MZ {uniq['all']['microcensus']:,}.")
# Vertical version of the chains plot (plot_total / draw_total stay untouched)
s, m = shares(sim_top, 'chain', chain_cats), shares(mz_top, 'chain', chain_cats)
bar_colors = [COLORS['pt']] * len(top) + [OTHER_COLOR]

x = np.arange(len(chain_cats))                   # first chain on the left
w, gap = 0.36, 0.04
fig, ax = plt.subplots(figsize=(0.9 * len(chain_cats) + 1.5, 5.5))

for offset, data, hatch in [(-(w + gap) / 2, s, None),      # simulation: left, solid
                            (+(w + gap) / 2, m, '///')]:    # Microcensus: right, hatched
    ax.bar(x + offset, data[chain_cats], w, color=bar_colors, hatch=hatch,
           edgecolor='white', linewidth=1.2)
    for xi, v in zip(x + offset, data[chain_cats]):
        ax.text(xi, v + 0.004, f'{v:.1%}', ha='center', va='bottom',
                rotation=90, fontsize=11, color=TEXT_COLOR)

ax.set_xticks(x, chain_cats, rotation=45, ha='right')
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.set_ylim(0, max(s[chain_cats].max(), m[chain_cats].max(), 0.01) * 1.2)
ax.set_ylabel('Share of all tours')
style_axes(ax, 'y')
ax.legend(handles=src_legend(), frameon=False, loc='upper right',
          handlelength=1.4, handleheight=1.4)
fig.text(0.0, -0.08, note, fontsize=11, color=TEXT_COLOR)
fig.tight_layout()
fig.savefig(OUT_DIR / 'chains_top.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# %% 13. Chain shares by sex (same chains and same order as the total figure) ----------------
SEXES = ['male', 'female']
per_sex = {}
for sex in SEXES:
    sim_s = group_other(sim_chains[sim_chains['sex_'] == sex], top)    # `top` from the total figure
    mz_s  = group_other(mz_chains[mz_chains['sex_'] == sex], top)
    per_sex[sex] = (sim_s, mz_s)
    res = compare(sim_s, mz_s, 'chain', chain_cats)
    res.index.name = 'chain'
    print(f'\n--- {nice_label(sex)} ---\n', res); res.to_csv(OUT_DIR / f'chains_{sex}.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 0.55 * len(chain_cats) + 1.8))
for ax, sex in zip(axes, SEXES):
    sim_s, mz_s = per_sex[sex]
    draw_total(ax, shares(sim_s, 'chain', chain_cats), shares(mz_s, 'chain', chain_cats), chain_cats,
               labels={c: c for c in chain_cats},
               colors={**{c: SEX_COLORS[sex] for c in top}, OTHER: OTHER_COLOR})
    u = uniq[sex]
    ax.set_title(f"{nice_label(sex)} ",
                 loc='left', fontsize=15, fontweight='bold')
    ax.set_xlabel(f'Share of all tours of {sex} persons')
axes[1].legend(handles=src_legend(), frameon=False, loc='lower right', handlelength=1.4, handleheight=1.4)
fig.text(0.0, -0.01, CHAIN_NOTE + "; 'Other' = all chains outside the top " + str(TOP_N_CHAINS)
         + ' of the total figure', fontsize=11, color=TEXT_COLOR)
fig.tight_layout()
fig.savefig(OUT_DIR / 'chains_by_sex.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# %% 14. Chain shares by income group (same chains and same order as the total figure) ----------
# Paste after the "chains by sex" cell. Needs: sim_chains, mz_chains, top, chain_cats, group_other,
# compare, shares, draw_total, src_legend, COLORS, OTHER, OTHER_COLOR, CHAIN_NOTE, TOP_N_CHAINS, OUT_DIR,
# and the income cell (read_matsim_income, sim_inc, mz_inc, read_mz, area_hh).

INCOME_GROUPS = {                                   # label -> income classes (MZ numbering 1-9)
    'Income classes 1-2 (< 4k CHF)':       [1, 2, 3, 4,5],
    'Income classes 7-8-9 (> 12k CHF)':    [ 6,7, 8, 9],
}
INCOME_GROUP_COLORS = {'Income classes 1-2 (< 4k CHF)': COLORS['bike'],
                       'Income classes 7-8-9 (> 12k CHF)': COLORS['walk']}


# --- 1) simulation: person -> household -> income class -------------------------------------
def read_matsim_members(path):
    """household id -> member person ids, from <household><members><personId refId=.../></members>."""
    rows, hh = [], None
    with gzip.open(path, 'rb') as f:
        for event, el in ET.iterparse(f, events=('start', 'end')):
            t = el.tag.rsplit('}', 1)[-1]
            if event == 'start':
                if t == 'household':
                    hh = el.get('id')
            else:
                if t == 'personId' and hh is not None:
                    rows.append((hh, el.get('refId')))
                elif t == 'household':
                    hh = None
                    el.clear()
    return pd.DataFrame(rows, columns=['hh', 'person'])


# income per household (already shifted +1 to MZ classes in `sim_inc`)
if unit == 'households':
    members = read_matsim_members(MATSIM_HH_FILE)
    hh_income = sim_inc.rename(columns={'id': 'hh'})[['hh', 'income']]
    person_income = members.merge(hh_income, on='hh', how='left')[['person', 'income']]
else:                                               # income stored directly on the persons
    person_income = sim_inc.rename(columns={'id': 'person'})[['person', 'income']]

sim_chains_inc = sim_chains.merge(person_income, on='person', how='left')
print(f"Simulation: {sim_chains_inc['income'].notna().mean():.1%} of the tours have an income class")


# --- 2) Microcensus: HHNR identifies both household and target person -----------------------
mz_income = mz_inc[['HHNR', 'income']] if 'HHNR' in mz_inc.columns else None
if mz_income is None:                               # mz_inc was filtered down to 'income'/'weight' only -> re-read
    tmp = read_mz('Haushalte.csv', ['HHNR'], optional=['F20601A', 'F20601'])
    col = 'F20601A' if 'F20601A' in tmp.columns else 'F20601'
    mz_income = tmp.assign(income=pd.to_numeric(tmp[col], errors='coerce'))[['HHNR', 'income']]
mz_chains_inc = mz_chains.merge(mz_income, on='HHNR', how='left')


# --- 3) tables + figure -----------------------------------------------------------------------
per_inc = {}
for label, classes in INCOME_GROUPS.items():
    sim_g = group_other(sim_chains_inc[sim_chains_inc['income'].isin(classes)], top)   # `top` = total figure
    mz_g  = group_other(mz_chains_inc[mz_chains_inc['income'].isin(classes)], top)
    per_inc[label] = (sim_g, mz_g)
    res = compare(sim_g, mz_g, 'chain', chain_cats)
    res.index.name = 'chain'
    print(f'\n--- {label} ---\n', res)
    res.to_csv(OUT_DIR / f"chains_income_{'_'.join(map(str, classes))}.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 0.55 * len(chain_cats) + 1.8))
for ax, (label, classes) in zip(axes, INCOME_GROUPS.items()):
    sim_g, mz_g = per_inc[label]
    draw_total(ax, shares(sim_g, 'chain', chain_cats), shares(mz_g, 'chain', chain_cats), chain_cats,
               labels={c: c for c in chain_cats},
               colors={**{c: INCOME_GROUP_COLORS[label] for c in top}, OTHER: OTHER_COLOR})
    ax.set_title(label, loc='left', fontsize=15, fontweight='bold')
    ax.set_xlabel('Share of all tours of persons in this income group')
axes[1].legend(handles=src_legend(), frameon=False, loc='lower right', handlelength=1.4, handleheight=1.4)
fig.text(0.0, -0.01, CHAIN_NOTE + "; 'Other' = all chains outside the top " + str(TOP_N_CHAINS)
         + ' of the total figure', fontsize=11, color=TEXT_COLOR)
fig.tight_layout()
fig.savefig(OUT_DIR / 'chains_by_income.png', dpi=200, bbox_inches='tight')
plt.show()

#### Activity chains by driving licence
Same chains and same order as the total figure (`top` / `chain_cats`). Left: persons **with** a driving licence, middle: persons **without**, right: difference in share between the two groups (licence − no licence, in percentage points), for the simulation (solid) and the Microcensus (hatched). Shares are computed within each group. A second figure restricts the comparison to persons aged 18+ (the age at which a licence can be held), so that children do not dominate the "no licence" group.

In [ ]:
# %% 15. Chain shares by driving licence (same chains and same order as the total figure) ------
# Needs: sim_chains, mz_chains, top, chain_cats, group_other, compare, shares, draw_total, src_legend, ...
LIC_GROUPS = {'yes': 'With driving licence', 'no': 'Without driving licence'}

# attach licence + age to every tour (simulation: person id, MZ: HHNR = one target person per household)
sim_chains_lic = sim_chains.merge(sim_p[['person', 'licence', 'age']], on='person', how='left')
mz_chains_lic  = mz_chains.merge(mz_pw[['HHNR', 'licence', 'age']], on='HHNR', how='left')


def chains_by_licence(min_age, fname, tag):
    """Chain shares for licence holders / non-holders aged `min_age`+ (None = everyone), 3 panels:
    licence | no licence | difference (licence - no licence, % points)."""
    per, sh = {}, {}
    for lic in LIC_GROUPS:
        sel_s = sim_chains_lic['licence'] == lic
        sel_m = mz_chains_lic['licence'] == lic
        if min_age is not None:
            sel_s &= sim_chains_lic['age'] >= min_age
            sel_m &= mz_chains_lic['age'] >= min_age
        sim_g, mz_g = group_other(sim_chains_lic[sel_s], top), group_other(mz_chains_lic[sel_m], top)
        per[lic] = (sim_g, mz_g)
        sh[lic] = (shares(sim_g, 'chain', chain_cats), shares(mz_g, 'chain', chain_cats))
        res = compare(sim_g, mz_g, 'chain', chain_cats)
        res.index.name = 'chain'
        print(f'\n--- {LIC_GROUPS[lic]} {tag} ---\n', res)
        res.to_csv(OUT_DIR / f'chains_licence_{lic}_{fname.replace(".png", "")}.csv')

    # difference between the two groups within each source (% points)
    dif = pd.DataFrame({
        'sim_licence': sh['yes'][0], 'sim_no_licence': sh['no'][0],
        'sim_diff_pp': (sh['yes'][0] - sh['no'][0]) * 100,
        'mz_licence': sh['yes'][1], 'mz_no_licence': sh['no'][1],
        'mz_diff_pp': (sh['yes'][1] - sh['no'][1]) * 100}).round(4)
    dif.index.name = 'chain'
    print(f'\n--- Difference licence - no licence {tag} ---\n', dif)
    dif.to_csv(OUT_DIR / f'chains_licence_difference_{fname.replace(".png", "")}.csv')

    fig, axes = plt.subplots(1, 3, figsize=(21, 0.55 * len(chain_cats) + 1.8), sharey=True)
    for ax, lic in zip(axes[:2], LIC_GROUPS):
        s, m = sh[lic]
        draw_total(ax, s, m, chain_cats, labels={c: c for c in chain_cats},
                   colors={**{c: LICENCE_COLORS[lic] for c in top}, OTHER: OTHER_COLOR})
        n_s, n_m = len(per[lic][0]), len(per[lic][1])
        ax.set_title(f'{LIC_GROUPS[lic]} {tag}', loc='left', fontsize=15, fontweight='bold')
        ax.set_xlabel(f'Share of all tours of this group\n(tours: simulation {n_s:,}, MZ {n_m:,} unweighted)')
    axes[1].legend(handles=src_legend(), frameon=False, loc='lower right', handlelength=1.4, handleheight=1.4)

    # third panel: difference in % points (same bar layout as draw_total)
    ax = axes[2]
    y = np.arange(len(chain_cats))[::-1]
    h, gap = 0.36, 0.04
    lim = max(dif['sim_diff_pp'].abs().max(), dif['mz_diff_pp'].abs().max(), 0.5)
    for offset, col, hatch in [(+(h + gap) / 2, 'sim_diff_pp', None), (-(h + gap) / 2, 'mz_diff_pp', '///')]:
        v = dif[col].reindex(chain_cats)
        ax.barh(y + offset, v, h, color=[LICENCE_COLORS['yes'] if c in top else OTHER_COLOR for c in chain_cats],
                hatch=hatch, edgecolor='white', linewidth=1.2)
        for yi, val in zip(y + offset, v):
            ax.text(val + (0.02 * lim if val >= 0 else -0.02 * lim), yi, f'{val:+.1f}', va='center',
                    ha='left' if val >= 0 else 'right', fontsize=11, color=TEXT_COLOR)
    ax.axvline(0, color=TEXT_COLOR, linewidth=0.8)
    ax.set_yticks(y, chain_cats)
    ax.set_xlim(-lim * 1.25, lim * 1.25)
    ax.set_xlabel('Difference in share (% points)\n> 0: more frequent with licence, < 0: more frequent without')
    ax.set_title(f'Difference (licence − no licence) {tag}', loc='left', fontsize=15, fontweight='bold')
    style_axes(ax, 'x')
    for a in axes:
        a.tick_params(labelleft=True)

    fig.text(0.0, -0.02, CHAIN_NOTE + "; 'Other' = all chains outside the top " + str(TOP_N_CHAINS)
             + ' of the total figure', fontsize=11, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()


chains_by_licence(None, 'chains_by_licence.png', f'(all ages {MIN_AGE}+)')
chains_by_licence(18,   'chains_by_licence_18plus.png', '(age 18+)')

#### Destination maps

In [ ]:
# %% 14. Destinations of work / education / shopping / leisure inside the study area (hexagonal bins)

MIN_MZ_SHARE = 0.5   # % of a purpose's MZ destinations a hexagon needs to be compared


from matplotlib.collections import PolyCollection

# simulation: all such activities of agents aged MIN_AGE+
sim_d = sim_acts.merge(sim_p[['person', 'age']], on='person')
sim_d['purpose'] = sim_d['type'].map(MATSIM_MAP)
sim_d = sim_d[sim_d['purpose'].isin(MAP_PURPOSES) & sim_d[['x', 'y']].notna().all(axis=1)]
sim_d = sim_d[inside(sim_d['x'], sim_d['y'])].assign(w=1.0)[['purpose', 'x', 'y', 'w', 'age']]

# Microcensus: weekday trips ending inside the study area
mz_d = mz_t[mz_t['letter'].isin(MAP_PURPOSES) & mz_t['dest_in']]
mz_d = mz_d.rename(columns={'letter': 'purpose', 'dx': 'x', 'dy': 'y', 'weight': 'w'})[['purpose', 'x', 'y', 'w', 'age']]
print(f'destinations inside the area: simulation {len(sim_d):,}, MZ {len(mz_d):,} (unweighted)')


def hex_cells(x, y, size, x0, y0):
    """Assign points to a pointy-top hexagonal grid (axial coordinates q, r).
    `size` = centre-to-corner distance in metres; (x0, y0) = grid origin."""
    x = np.asarray(x, dtype=float) - x0
    y = np.asarray(y, dtype=float) - y0
    q = (np.sqrt(3) / 3 * x - y / 3) / size
    r = (2 / 3 * y) / size
    # cube rounding
    cx, cz = q, r
    cy = -cx - cz
    rx, ry, rz = np.round(cx), np.round(cy), np.round(cz)
    dx, dy, dz = np.abs(rx - cx), np.abs(ry - cy), np.abs(rz - cz)
    m1 = (dx > dy) & (dx > dz)
    m2 = ~m1 & (dy > dz)
    q_out = np.where(m1, -ry - rz, rx)
    r_out = np.where(~m1 & ~m2, -rx - ry, rz)
    return q_out.astype(int), r_out.astype(int)


def hex_centres(q, r, size, x0, y0):
    """Axial coordinates -> hexagon centres in map coordinates."""
    cx = size * np.sqrt(3) * (q + r / 2) + x0
    cy = size * 1.5 * r + y0
    return cx, cy


def plot_destination_maps(sim_d, mz_d, cell_m, age_range, fname):
    """Rows: simulation, Microcensus, difference. Columns: purposes. Each hexagon shows the % of that
    purpose's destinations falling into it, so the sources are comparable despite different sizes.
    `cell_m` = width of a hexagon (distance between two opposite flat sides).
    Colour scales are shared: one for rows 1-2 (all purposes), one for the difference row."""
    lo, hi = age_range
    def select(df):
        keep = np.ones(len(df), dtype=bool)
        if lo is not None: keep &= (df['age'] >= lo).to_numpy()
        if hi is not None: keep &= (df['age'] <= hi).to_numpy()
        return df[keep]
    sources = {'Simulation': select(sim_d), 'Microcensus': select(mz_d)}

    minx, miny, maxx, maxy = area.total_bounds
    size = cell_m / np.sqrt(3)                       # centre-to-corner distance
    angles = np.deg2rad(30 + 60 * np.arange(6))      # pointy-top hexagon corners
    corners = size * np.column_stack([np.cos(angles), np.sin(angles)])

    def hex_collection(values, cmap, vmin, vmax, alpha, norm=None):
        """PolyCollection with one hexagon per (q, r) in the Series `values`."""
        q = values.index.get_level_values(0).to_numpy()
        r = values.index.get_level_values(1).to_numpy()
        cx, cy = hex_centres(q, r, size, minx, miny)
        verts = np.stack([cx, cy], axis=1)[:, None, :] + corners[None, :, :]
        pc = PolyCollection(verts, array=values.to_numpy(), cmap=cmap, alpha=alpha,
                            edgecolors='white', linewidths=0.2)
        if norm is not None:
            pc.set_norm(norm)
        else:
            pc.set_clim(vmin, vmax)
        return pc

    def decorate(ax):
        area.boundary.plot(ax=ax, color='black', linewidth=1)
        ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
        try:                                       # swisstopo tiles need internet access
            ctx.add_basemap(ax, source=SWISSTOPO_TILES, crs=area.crs, zoom=MAP_ZOOM, attribution=False)
        except Exception as e:
            print('basemap not available:', type(e).__name__)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(''); ax.set_ylabel('')

    # --- 1) bin every source / purpose into hexagons (% of that purpose's destinations) ---
    grids, totals, diffs = {}, {}, {}
    for code in MAP_PURPOSES:
        for src, df in sources.items():
            d = df[df['purpose'] == code]
            totals[code, src] = d['w'].sum()
            if len(d) == 0:
                grids[code, src] = pd.Series(dtype=float, index=pd.MultiIndex.from_arrays([[], []]))
                continue
            q, r = hex_cells(d['x'], d['y'], size, minx, miny)
            s = d['w'].groupby([q, r]).sum()
            grids[code, src] = 100 * s / s.sum()
        # relative difference (Sim - MZ) / MZ in %, MZ = base.
        # Only hexagons holding at least MIN_MZ_SHARE % of the MZ destinations are compared.
        sim_g, mz_g = grids[code, 'Simulation'], grids[code, 'Microcensus']
        mz_g = mz_g[mz_g >= MIN_MZ_SHARE]
        diffs[code] = 100 * (sim_g.reindex(mz_g.index, fill_value=0) - mz_g) / mz_g

    vmax = max((g.max() for g in grids.values() if len(g)), default=1.0)          # one scale for rows 1-2
    #lim = max((d.abs().max() for d in diffs.values() if len(d)), default=1.0)   # symmetric, from the data
    lim = max((d.max() for d in diffs.values() if len(d)), default=100.0)   # largest positive value
    dnorm = TwoSlopeNorm(vcenter=0, vmin=-100, vmax=max(lim, 1e-6))

    # --- 2) plot ---
    fig, axes = plt.subplots(3, len(MAP_PURPOSES), figsize=(20, 15), layout='constrained')
    mesh = dmesh = None
    for j, (code, name) in enumerate(MAP_PURPOSES.items()):
        for i, src in enumerate(sources):
            ax = axes[i, j]
            mesh = hex_collection(grids[code, src], SEQ_CMAP, 0, vmax, 0.8)
            ax.add_collection(mesh)
            decorate(ax)
            ax.set_xlabel(f'{totals[code, src]:,.0f} destinations', fontsize=14)
            if i == 0: ax.set_title(name, fontsize=16, fontweight='bold')
            if j == 0: ax.set_ylabel(src, fontsize=14, fontweight='bold')

        ax = axes[2, j]
        dmesh = hex_collection(diffs[code], DIV_CMAP, None, None, 0.9, norm=dnorm)
        ax.add_collection(dmesh)
        decorate(ax)
        if j == 0: ax.set_ylabel('Relative difference\n(Simulation − MZ) / MZ', fontsize=14, fontweight='bold')

    # --- 3) two shared colourbars on the right ---
    cbar = fig.colorbar(mesh, ax=axes[:2, :], location='right', shrink=0.8, pad=0.01,
                        label='% of destinations per hexagon')
    cbar.ax.yaxis.label.set_size(14); cbar.ax.tick_params(labelsize=12)

    cbar = fig.colorbar(dmesh, ax=axes[2, :], location='right', shrink=0.8, pad=0.01,
                        label='relative difference in %\n(coral: more in simulation,\nblue: more in MZ)')
    cbar.ax.yaxis.label.set_size(12); cbar.ax.tick_params(labelsize=12)

    age_label = f', age {lo}–{hi}' if lo is not None or hi is not None else ''
    fig.suptitle(f'Where people go: Simulation vs Microcensus (hexagons {cell_m} m wide{age_label})',
                 fontsize=18, fontweight='bold')
    fig.text(0.995, 0.005, 'Background: \u00a9 swisstopo', ha='right', fontsize=8)
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.show()


for run in MAP_RUNS:
    lo, hi = run['age_range']
    tag = 'all_ages' if lo is None and hi is None else f'age{lo}-{hi}'
    plot_destination_maps(sim_d, mz_d, run['cell_m'], run['age_range'], f'destination_maps_hex_{tag}_{run["cell_m"]}m.png')

#### Car availability

In [ ]:
# %% carAvail: which values exist in the population file? --------------------------

def count_attribute(path, attr_name='carAvail'):
    """Stream the plans file once and count every value of person attribute `attr_name`.
    Persons without the attribute are counted as '<missing>'."""
    counts = Counter()
    with gzip.open(path, 'rb') as f:
        for _, el in ET.iterparse(f, events=('end',)):
            if el.tag != 'person':
                continue
            value = None
            for a in el.findall('attributes/attribute'):
                if a.get('name') == attr_name:
                    value = (a.text or '').strip()
                    break
            counts[value if value is not None else '<missing>'] += 1
            el.clear()                                   # free memory
    return counts

car_counts = count_attribute(MATSIM_POP_FILE, 'carAvail')
car_df = (pd.Series(car_counts, name='n_persons')
          .rename_axis('carAvail').reset_index()
          .sort_values('n_persons', ascending=False))
car_df['share'] = car_df['n_persons'] / car_df['n_persons'].sum()

print(f"Persons in file: {car_df['n_persons'].sum():,}")
print(car_df.to_string(index=False, formatters={'share': '{:.1%}'.format}))
car_df.to_csv(OUT_DIR / 'carAvail_values.csv', index=False)

In [ ]:
# %% 15. Car availability: attach to simulation persons and MZ trips -------------------
# --- Simulation: carAvail per person (one extra pass over the plans file) ---
def read_matsim_caravail(path):
    out = {}
    with gzip.open(path, 'rb') as f:
        for _, el in ET.iterparse(f, events=('end',)):
            if el.tag != 'person':
                continue
            val = None
            for a in el.findall('attributes/attribute'):
                if a.get('name') == 'carAvail':
                    val = (a.text or '').strip().lower()
                    break
            out[el.get('id')] = val
            el.clear()
    return pd.Series(out)

sim_p['car'] = sim_p['person'].map(read_matsim_caravail(MATSIM_POP_FILE))

# --- Microcensus: f42100e per target person ---
# 1 = always, 2 = by arrangement (left out), 3 = never,
# -99 = under 18 or no driving licence -> counted as 'never' to match the simulation
MZ_CAR_CODES = {1: 'always', 3: 'never', -99: 'never'}     # remove -99 for a strict "3 = never"
mz_car = read_mz('zielpersonen.csv', ['HHNR', 'f42100e']).drop_duplicates('HHNR').set_index('HHNR')['f42100e']
mz_t['car'] = mz_t['HHNR'].map(mz_car).map(MZ_CAR_CODES)

# --- sanity checks ---
print('Simulation, carAvail x licence (persons):')
print(pd.crosstab(sim_p['car'].fillna('<missing>'), sim_p['licence'].fillna('<missing>')))
print('\nMZ weekday trips by car availability:')
print(mz_t['car'].fillna('other / sometimes').value_counts().to_string())

# --- destination tables with the car group (same filters as section 14) ---
sim_dc = sim_acts.merge(sim_p[['person', 'age', 'car']], on='person')
sim_dc['purpose'] = sim_dc['type'].map(MATSIM_MAP)
sim_dc = sim_dc[sim_dc['purpose'].isin(MAP_PURPOSES) & sim_dc[['x', 'y']].notna().all(axis=1)]
sim_dc = sim_dc[inside(sim_dc['x'], sim_dc['y'])].assign(w=1.0)[['purpose', 'x', 'y', 'w', 'age', 'car']]

mz_dc = mz_t[mz_t['letter'].isin(MAP_PURPOSES) & mz_t['dest_in']]
mz_dc = mz_dc.rename(columns={'letter': 'purpose', 'dx': 'x', 'dy': 'y', 'weight': 'w'})[
    ['purpose', 'x', 'y', 'w', 'age', 'car']]

In [ ]:
# %% 17. Destination map: car always available, Simulation vs Microcensus ---------------
sim_always = sim_dc[sim_dc['car'] == 'always'].drop(columns='car')
mz_always  = mz_dc[mz_dc['car'] == 'always'].drop(columns='car')
print(f'car always available, destinations in the area: '
      f'simulation {len(sim_always):,}, MZ {len(mz_always):,} (unweighted)')

plot_destination_maps(sim_always, mz_always, cell_m=500, age_range=(None, None),
                      fname='destination_maps_car_always_500m.png')

In [ ]:
# %% 14. Destinations by municipality (choropleth) --------------------------------------
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

MUNI_FILE = (PROJECT_DIR / "Sylvain" / "working_rep" / "vaud_limites" / "limites_officielles"
             / "LAD_MO_COMMUNE.shp")
VALUE_MODE = 'share'      # 'share' = % of destinations per municipality
                          # 'density' = % of destinations per km² (removes the effect of municipality size)
LABEL_MUNIS = True        # write municipality names on the difference row (if a name column is found)

# --- municipalities: LV95, only those touching the study area ---
munis = gpd.read_file(MUNI_FILE)
munis = munis.set_crs(2056) if munis.crs is None else munis.to_crs(2056)
munis = munis[munis.intersects(shape)].reset_index(drop=True)
munis['mid'] = munis.index
munis['area_km2'] = munis.geometry.area / 1e6
print('shapefile columns:', [c for c in munis.columns if c != 'geometry'])

NAME_COL = next((c for c in munis.columns
                 if c.lower() in ('nom', 'name', 'commune', 'nom_commune', 'nom_comm', 'gemeindename',
                                  'gemeinde', 'nom_officiel', 'name_de', 'bfs_name')), None)
print(f'{len(munis)} municipalities touch the study area; name column: {NAME_COL}')   # if None: set NAME_COL by hand


def assign_muni(df):
    """Add column 'mid' = index of the municipality containing each destination (NaN if none)."""
    df = df.reset_index(drop=True)
    pts = gpd.GeoDataFrame(df[['x']].copy(), geometry=gpd.points_from_xy(df['x'], df['y']), crs=2056)
    j = gpd.sjoin(pts, munis[['mid', 'geometry']], how='left', predicate='within')
    j = j[~j.index.duplicated()]                       # a point can only be counted once
    out = df.assign(mid=j['mid'].reindex(df.index).to_numpy())
    n_out = out['mid'].isna().sum()
    if n_out:
        print(f'  {n_out:,} destinations not in any municipality (dropped)')
    return out.dropna(subset=['mid']).astype({'mid': int})


sim_dm = assign_muni(sim_d)
mz_dm  = assign_muni(mz_d)
print(f'destinations assigned to a municipality: simulation {len(sim_dm):,}, MZ {len(mz_dm):,} (unweighted)')


def plot_muni_maps(sim_d, mz_d, age_range, fname):
    """Rows: simulation, Microcensus, difference. Columns: purposes. Each municipality shows the % of that
    purpose's destinations (or % per km² if VALUE_MODE = 'density'). Shared colour scales as before."""
    lo, hi = age_range
    def select(df):
        keep = np.ones(len(df), dtype=bool)
        if lo is not None: keep &= (df['age'] >= lo).to_numpy()
        if hi is not None: keep &= (df['age'] <= hi).to_numpy()
        return df[keep]
    sources = {'Simulation': select(sim_d), 'Microcensus': select(mz_d)}

    minx, miny, maxx, maxy = area.total_bounds
    unit = '% of destinations per km²' if VALUE_MODE == 'density' else '% of destinations per municipality'

    def decorate(ax):
        munis.boundary.plot(ax=ax, color='#8a8984', linewidth=0.4)
        area.boundary.plot(ax=ax, color='black', linewidth=1)
        ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
        try:
            ctx.add_basemap(ax, source=SWISSTOPO_TILES, crs=area.crs, zoom=MAP_ZOOM, attribution=False)
        except Exception as e:
            print('basemap not available:', type(e).__name__)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

    # --- 1) value per municipality, per source and purpose ---
    vals, totals, diffs = {}, {}, {}
    for code in MAP_PURPOSES:
        for src, df in sources.items():
            d = df[df['purpose'] == code]
            totals[code, src] = d['w'].sum()
            s = d.groupby('mid')['w'].sum().reindex(munis['mid']).fillna(0)
            v = 100 * s / s.sum() if s.sum() > 0 else s
            if VALUE_MODE == 'density':
                v = v / munis['area_km2']
            vals[code, src] = v
        diffs[code] = vals[code, 'Simulation'] - vals[code, 'Microcensus']    # percentage points

    vmax = max(v.max() for v in vals.values()) or 1.0
    lim = max(d.abs().max() for d in diffs.values()) or 1.0
    seq_norm, div_norm = Normalize(0, vmax), Normalize(-lim, lim)

    def draw(ax, v, cmap, norm, alpha):
        g = munis.assign(v=v.to_numpy())
        if cmap is SEQ_CMAP:
            g = g[g['v'] > 0]                          # no destinations -> leave unfilled
        g.plot(column='v', cmap=cmap, norm=norm, ax=ax, alpha=alpha, edgecolor='none')

    # --- 2) plot ---
    fig, axes = plt.subplots(3, len(MAP_PURPOSES), figsize=(20, 15), layout='constrained')
    for j, (code, name) in enumerate(MAP_PURPOSES.items()):
        for i, src in enumerate(sources):
            ax = axes[i, j]
            draw(ax, vals[code, src], SEQ_CMAP, seq_norm, 0.8)
            decorate(ax)
            ax.set_xlabel(f'{totals[code, src]:,.0f} destinations', fontsize=14)
            if i == 0: ax.set_title(name, fontsize=16, fontweight='bold')
            if j == 0: ax.set_ylabel(src, fontsize=14, fontweight='bold')

        ax = axes[2, j]
        draw(ax, diffs[code], DIV_CMAP, div_norm, 0.9)
        decorate(ax)
        if LABEL_MUNIS and NAME_COL:
            for _, m in munis.iterrows():
                p = m.geometry.representative_point()
                if minx <= p.x <= maxx and miny <= p.y <= maxy:
                    ax.annotate(str(m[NAME_COL]), (p.x, p.y), ha='center', fontsize=7, color=TEXT_COLOR)
        if j == 0: ax.set_ylabel('Difference\n(Simulation − MZ)', fontsize=14, fontweight='bold')

    # --- 3) two shared colourbars ---
    cbar = fig.colorbar(ScalarMappable(seq_norm, SEQ_CMAP), ax=axes[:2, :], location='right',
                        shrink=0.8, pad=0.01, label=unit)
    cbar.ax.yaxis.label.set_size(14); cbar.ax.tick_params(labelsize=12)
    cbar = fig.colorbar(ScalarMappable(div_norm, DIV_CMAP), ax=axes[2, :], location='right',
                        shrink=0.8, pad=0.01,
                        label='difference in % points' + (' per km²' if VALUE_MODE == 'density' else '')
                              + '\n(coral: more in simulation,\nblue: more in MZ)')
    cbar.ax.yaxis.label.set_size(12); cbar.ax.tick_params(labelsize=12)

    age_label = f', age {lo}–{hi}' if lo is not None or hi is not None else ''
    fig.suptitle(f'Where people go: Simulation vs Microcensus (municipalities{age_label})',
                 fontsize=18, fontweight='bold')
    fig.text(0.995, 0.005, 'Background: \u00a9 swisstopo', ha='right', fontsize=8)
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.show()

    # table with the numbers behind the maps
    rows = []
    for code in MAP_PURPOSES:
        t = pd.DataFrame({'purpose': code, 'municipality': munis[NAME_COL] if NAME_COL else munis['mid'],
                          'sim': vals[code, 'Simulation'], 'mz': vals[code, 'Microcensus'],
                          'diff': diffs[code]})
        rows.append(t)
    pd.concat(rows).round(3).to_csv(OUT_DIR / fname.replace('.png', '.csv'), index=False)



In [ ]:

for run in MAP_RUNS:
    lo, hi = run['age_range']
    tag = 'all_ages' if lo is None and hi is None else f'age{lo}-{hi}'
    plot_muni_maps(sim_dm, mz_dm, run['age_range'], f'destination_maps_muni_{VALUE_MODE}_{tag}.png')

#### Shopping destinations: car available vs no car
Weekday shopping destinations inside the study area (hexagons, % of the group's shopping destinations per hexagon). Car group as defined above: `always` = car always available, `never` = never (incl. persons without a licence, as in the previous section); persons with "sometimes / by arrangement" are left out.

* Rows 1-2: Simulation and Microcensus; columns 1-2: car always available / no car. Row 3: relative difference Simulation vs Microcensus (unchanged from the other destination maps, hexagons with at least `MIN_MZ_SHARE` % of the MZ destinations).
* Column 3 (rows 1-2): **impact of car ownership within one model**: share with car − share without car, in percentage points, for the simulation and for the Microcensus separately (coral: relatively more shopping destinations in this hexagon for people with a car, blue: more for people without).

In [ ]:
# %% 18. Shopping destinations by car availability: Simulation vs MZ and car vs no car ----------
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
CAR_GROUPS = {'always': 'Car always available', 'never': 'No car available'}


def plot_shopping_car_maps(sim_dc, mz_dc, cell_m, fname, purpose='S'):
    """3x3 figure. Rows: Simulation, Microcensus, relative difference Sim vs MZ (per car group).
    Columns: car always / no car / impact of car (car - no car, % points, rows 1-2 only)."""
    sources = {'Simulation': sim_dc, 'Microcensus': mz_dc}
    minx, miny, maxx, maxy = area.total_bounds
    size = cell_m / np.sqrt(3)                       # centre-to-corner distance
    angles = np.deg2rad(30 + 60 * np.arange(6))
    corners = size * np.column_stack([np.cos(angles), np.sin(angles)])

    def hex_collection(values, cmap, alpha, norm):
        q = values.index.get_level_values(0).to_numpy()
        r = values.index.get_level_values(1).to_numpy()
        cx, cy = hex_centres(q, r, size, minx, miny)
        verts = np.stack([cx, cy], axis=1)[:, None, :] + corners[None, :, :]
        pc = PolyCollection(verts, array=values.to_numpy(), cmap=cmap, alpha=alpha,
                            edgecolors='white', linewidths=0.2)
        pc.set_norm(norm)
        return pc

    def draw(ax, values, cmap, alpha, norm):
        if len(values):
            ax.add_collection(hex_collection(values, cmap, alpha, norm))
        area.boundary.plot(ax=ax, color='black', linewidth=1)
        ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
        try:
            ctx.add_basemap(ax, source=SWISSTOPO_TILES, crs=area.crs, zoom=MAP_ZOOM, attribution=False)
        except Exception as e:
            print('basemap not available:', type(e).__name__)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

    # --- 1) % of the group's shopping destinations per hexagon ---
    grids, totals = {}, {}
    for src, df in sources.items():
        for g in CAR_GROUPS:
            d = df[(df['purpose'] == purpose) & (df['car'] == g)]
            totals[src, g] = d['w'].sum()
            if len(d) == 0:
                grids[src, g] = pd.Series(dtype=float, index=pd.MultiIndex.from_arrays([[], []]))
                continue
            q, r = hex_cells(d['x'], d['y'], size, minx, miny)
            s = d['w'].groupby([q, r]).sum()
            grids[src, g] = 100 * s / s.sum()

    # --- 2) Simulation vs MZ, per car group (relative difference, MZ = base, as in the other maps) ---
    diff_src = {}
    for g in CAR_GROUPS:
        sim_g, mz_g = grids['Simulation', g], grids['Microcensus', g]
        mz_g = mz_g[mz_g >= MIN_MZ_SHARE]
        diff_src[g] = 100 * (sim_g.reindex(mz_g.index, fill_value=0) - mz_g) / mz_g

    # --- 3) impact of car ownership within one source (% points, car - no car) ---
    diff_car = {}
    for src in sources:
        a, b = grids[src, 'always'], grids[src, 'never']
        idx = a.index.union(b.index)
        diff_car[src] = a.reindex(idx, fill_value=0) - b.reindex(idx, fill_value=0)

    vmax = max((g.max() for g in grids.values() if len(g)), default=1.0)
    seq_norm = Normalize(0, vmax)
    lim = max((d.max() for d in diff_src.values() if len(d)), default=100.0)
    rel_norm = TwoSlopeNorm(vcenter=0, vmin=-100, vmax=max(lim, 1e-6))
    lim_car = max((d.abs().max() for d in diff_car.values() if len(d)), default=1.0)
    car_norm = Normalize(-max(lim_car, 1e-6), max(lim_car, 1e-6))

    # --- 4) plot ---
    fig, axes = plt.subplots(3, 3, figsize=(20, 19), layout='constrained')
    for i, src in enumerate(sources):
        for j, g in enumerate(CAR_GROUPS):
            ax = axes[i, j]
            draw(ax, grids[src, g], SEQ_CMAP, 0.8, seq_norm)
            ax.set_xlabel(f'{totals[src, g]:,.0f} shopping destinations', fontsize=14)
            if i == 0: ax.set_title(CAR_GROUPS[g], fontsize=16, fontweight='bold')
            if j == 0: ax.set_ylabel(src, fontsize=14, fontweight='bold')
        ax = axes[i, 2]                                      # impact of car within this source
        draw(ax, diff_car[src], DIV_CMAP, 0.9, car_norm)
        if i == 0: ax.set_title('Impact of car ownership\n(car − no car)', fontsize=16, fontweight='bold')
        ax.set_xlabel(f'within {src}', fontsize=14)

    for j, g in enumerate(CAR_GROUPS):                       # row 3: Simulation vs MZ (unchanged)
        ax = axes[2, j]
        draw(ax, diff_src[g], DIV_CMAP, 0.9, rel_norm)
        if j == 0: ax.set_ylabel('Relative difference\n(Simulation − MZ) / MZ', fontsize=14, fontweight='bold')
    axes[2, 2].axis('off')

    # --- 5) colourbars ---
    cbar = fig.colorbar(ScalarMappable(seq_norm, SEQ_CMAP), ax=axes[:2, :2], location='right',
                        shrink=0.8, pad=0.01, label='% of shopping destinations per hexagon')
    cbar.ax.yaxis.label.set_size(14); cbar.ax.tick_params(labelsize=12)
    cbar = fig.colorbar(ScalarMappable(rel_norm, DIV_CMAP), ax=axes[2, :2], location='right',
                        shrink=0.8, pad=0.01,
                        label='relative difference in %\n(coral: more in simulation,\nblue: more in MZ)')
    cbar.ax.yaxis.label.set_size(12); cbar.ax.tick_params(labelsize=12)
    cbar = fig.colorbar(ScalarMappable(car_norm, DIV_CMAP), ax=axes[:2, 2], location='right',
                        shrink=0.8, pad=0.01,
                        label='difference in % points\n(coral: more with car,\nblue: more without car)')
    cbar.ax.yaxis.label.set_size(12); cbar.ax.tick_params(labelsize=12)

    fig.suptitle(f'Where people go shopping, by car availability (hexagons {cell_m} m wide)',
                 fontsize=18, fontweight='bold')
    fig.text(0.995, 0.005, 'Background: \u00a9 swisstopo', ha='right', fontsize=8)
    fig.savefig(OUT_DIR / fname, dpi=200)
    plt.show()


In [ ]:


print('shopping destinations in the area (unweighted): ' + ', '.join(
    f'{name} {len(df[(df["purpose"] == "S") & (df["car"] == g)]):,} ({g})'
    for name, df in [('simulation', sim_dc), ('MZ', mz_dc)] for g in CAR_GROUPS))
plot_shopping_car_maps(sim_dc, mz_dc, cell_m=500, fname='destination_maps_shopping_car_500m.png')